In [ ]:
# ============================================================
# BASELINE LOSO EEG MODELING
# CNN-only + CNN-feature-fusion
# Arithmetic only: low / mid / high
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, random, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch
from scipy.stats import entropy

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ============================================================
# CONFIG
# ============================================================

Mounted at /content/drive


In [ ]:

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

SAVE_DIR = "/content/drive/MyDrive/eeg_qc_outputs"
NPZ_PATH = os.path.join(SAVE_DIR, "clean_eeg_windows_qc.npz")

FS = 250
N_CLASSES = 3
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
FEW_SHOT_PER_CLASS = 5

LABEL_REMAP = {
    "lowlevel": 0,
    "midlevel": 1,
    "highlevel": 2
}


DEVICE: cuda


In [ ]:
ID_TO_LABEL = {
    0: "lowlevel",
    1: "midlevel",
    2: "highlevel"
}


In [ ]:
data = np.load(NPZ_PATH, allow_pickle=True)

X = data["X"]              # (N, 1000, 8)
labels = data["labels"]    # strings
subjects = data["subjects"]
tasks = data["tasks"]
file_paths = data["file_paths"]
start_secs = data["start_secs"]

print("Loaded X:", X.shape)
print("Unique tasks:", np.unique(tasks))
print("Unique labels:", np.unique(labels))
print("Unique subjects:", np.unique(subjects))

# Arithmetic only, remove natural
mask = (
    (tasks == "Arithmetic") &
    np.isin(labels, ["lowlevel", "midlevel", "highlevel"])
)


Loaded X: (2371, 1000, 8)
Unique tasks: ['Arithmetic' 'Stroop']
Unique labels: ['highlevel' 'lowlevel' 'midlevel' 'natural']
Unique subjects: [ 1  2  3  5  6  7  8  9 10 11 12 13 14 15]


In [ ]:

X = X[mask]
labels = labels[mask]
subjects = subjects[mask]
file_paths = file_paths[mask]
start_secs = start_secs[mask]

y = np.array([LABEL_REMAP[l] for l in labels])

print("\nAfter Arithmetic non-natural filter:")
print("X:", X.shape)
print("Label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})
print("Subjects:", np.unique(subjects))


After Arithmetic non-natural filter:
X: (879, 1000, 8)
Label counts: {'lowlevel': 240, 'midlevel': 263, 'highlevel': 376}
Subjects: [ 1  2  3  5  6  7  8  9 10 11 12 13 14 15]


In [ ]:
# Convert from (N, T, C) to float32
X = X.astype(np.float32)

# ============================================================
# HANDCRAFTED FEATURES
# ============================================================

FEATURE_CACHE = os.path.join(SAVE_DIR, "arithmetic_baseline_features.npy")

bands = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 35),
}

In [ ]:
def bandpower_from_psd(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return np.trapezoid(pxx[idx], f[idx])

def spectral_entropy_from_psd(pxx):
    p = pxx / (np.sum(pxx) + 1e-12)
    return entropy(p)

In [ ]:
def extract_features_one_window(w, fs=250):
    """
    w: (T, C)
    Returns simple EEG features.
    """
    feats = []

    # time-domain stats per channel
    ch_mean = np.mean(w, axis=0)
    ch_std = np.std(w, axis=0)
    ch_var = np.var(w, axis=0)
    ch_ptp = np.ptp(w, axis=0)

    feats.extend(ch_mean.tolist())
    feats.extend(ch_std.tolist())
    feats.extend(ch_var.tolist())
    feats.extend(ch_ptp.tolist())

    # global stats
    feats.append(np.mean(ch_var))
    feats.append(np.max(ch_var))
    feats.append(np.mean(ch_ptp))
    feats.append(np.max(ch_ptp))

    # spectral features
    band_means = {b: [] for b in bands}
    spec_ents = []

    for ch in range(w.shape[1]):
        f, pxx = welch(w[:, ch], fs=fs, nperseg=512)

        for name, (lo, hi) in bands.items():
            bp = bandpower_from_psd(f, pxx, lo, hi)
            band_means[name].append(bp)

        spec_ents.append(spectral_entropy_from_psd(pxx))

    # per-band mean logpower
    for name in ["theta", "alpha", "beta", "gamma"]:
        vals = np.array(band_means[name])
        feats.append(np.mean(vals))
        feats.append(np.log(np.mean(vals) + 1e-12))
        feats.append(np.std(vals))

    theta = np.mean(band_means["theta"])
    alpha = np.mean(band_means["alpha"])
    beta = np.mean(band_means["beta"])
    gamma = np.mean(band_means["gamma"])

    feats.append(theta / (alpha + 1e-12))
    feats.append(theta / (beta + 1e-12))
    feats.append(alpha / (beta + 1e-12))
    feats.append(gamma / (beta + 1e-12))

    feats.append(np.log(theta + 1e-12) - np.log(alpha + 1e-12))
    feats.append(np.log(theta + 1e-12) - np.log(beta + 1e-12))
    feats.append(np.log(alpha + 1e-12) - np.log(beta + 1e-12))

    feats.append(np.mean(spec_ents))
    feats.append(np.std(spec_ents))

    return np.array(feats, dtype=np.float32)


In [ ]:
if os.path.exists(FEATURE_CACHE):
    print("\nLoading cached handcrafted features...")
    F_all = np.load(FEATURE_CACHE)
else:
    print("\nExtracting handcrafted features...")
    feats = []
    for i in range(len(X)):
        if i % 100 == 0:
            print(f"  extracting {i}/{len(X)}")
        feats.append(extract_features_one_window(X[i], FS))
    F_all = np.stack(feats).astype(np.float32)
    np.save(FEATURE_CACHE, F_all)
    print("Saved features:", FEATURE_CACHE)

print("Feature matrix:", F_all.shape)



Loading cached handcrafted features...
Feature matrix: (879, 57)


In [ ]:
# ============================================================
# DATASET
# ============================================================

class EEGDataset(Dataset):
    def __init__(self, X, y, features=None):
        self.X = X
        self.y = y.astype(np.int64)
        self.features = features

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # X originally (T, C), model wants (C, T)
        x = torch.tensor(self.X[idx].T, dtype=torch.float32)
        label = torch.tensor(self.y[idx], dtype=torch.long)

        if self.features is None:
            return x, label
        else:
            f = torch.tensor(self.features[idx], dtype=torch.float32)
            return x, f, label

def make_loader(X, y, features=None, batch_size=64, shuffle=False, weighted=False):
    ds = EEGDataset(X, y, features)

    if weighted:
        class_counts = np.bincount(y, minlength=N_CLASSES)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[y]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=False)

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

In [ ]:

# ============================================================
# MODELS
# ============================================================

class EEGCNN(nn.Module):
    def __init__(self, n_channels=8, n_classes=3, emb_dim=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.proj = nn.Sequential(
            nn.Linear(128, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x):
        h = self.encoder(x)
        h = self.pool(h).squeeze(-1)
        z = self.proj(h)
        return z

    def forward(self, x):
        z = self.forward_features(x)
        logits = self.classifier(z)
        return logits, z


In [ ]:
class EEGCNNFeatureFusion(nn.Module):
    def __init__(self, n_channels=8, n_features=1, n_classes=3, emb_dim=128, feat_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.raw_proj = nn.Sequential(
            nn.Linear(128, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.feat_proj = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, feat_dim),
            nn.GELU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(emb_dim + feat_dim, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x, features):
        h = self.encoder(x)
        h = self.pool(h).squeeze(-1)

        z_raw = self.raw_proj(h)
        z_feat = self.feat_proj(features)

        z = torch.cat([z_raw, z_feat], dim=1)
        z = self.fusion(z)
        return z

    def forward(self, x, features):
        z = self.forward_features(x, features)
        logits = self.classifier(z)
        return logits, z

In [ ]:
# ============================================================
# EVAL HELPERS
# ============================================================

@torch.no_grad()
def eval_classifier_cnn(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for x, yb in loader:
        x = x.to(device)
        yb = yb.to(device)

        logits, _ = model(x)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds, labels=[0,1,2])
    return acc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def eval_classifier_fusion(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for x, f, yb in loader:
        x = x.to(device)
        f = f.to(device)
        yb = yb.to(device)

        logits, _ = model(x, f)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds, labels=[0,1,2])
    return acc, cm, np.array(all_true), np.array(all_preds)

In [ ]:
@torch.no_grad()
def get_embeddings_cnn(model, loader):
    model.eval()
    Z, Y = [], []

    for x, yb in loader:
        x = x.to(device)
        _, z = model(x)
        Z.append(z.cpu())
        Y.append(yb)

    return torch.cat(Z, dim=0), torch.cat(Y, dim=0)

@torch.no_grad()
def get_embeddings_fusion(model, loader):
    model.eval()
    Z, Y = [], []

    for x, f, yb in loader:
        x = x.to(device)
        f = f.to(device)
        _, z = model(x, f)
        Z.append(z.cpu())
        Y.append(yb)

    return torch.cat(Z, dim=0), torch.cat(Y, dim=0)

In [ ]:
def prototype_eval(Z_support, y_support, Z_query, y_query):
    """
    Nearest prototype classifier in embedding space.
    """
    Z_support = F.normalize(Z_support, dim=1)
    Z_query = F.normalize(Z_query, dim=1)

    prototypes = []
    for c in range(N_CLASSES):
        mask = (y_support == c)
        if mask.sum() == 0:
            proto = torch.zeros(Z_support.shape[1])
        else:
            proto = Z_support[mask].mean(dim=0)
            proto = F.normalize(proto, dim=0)
        prototypes.append(proto)

    prototypes = torch.stack(prototypes, dim=0)

    sims = Z_query @ prototypes.T
    preds = sims.argmax(dim=1)

    acc = accuracy_score(y_query.numpy(), preds.numpy())
    cm = confusion_matrix(y_query.numpy(), preds.numpy(), labels=[0,1,2])
    return acc, cm, preds.numpy()


In [ ]:
# ============================================================
# SPLITTING
# ============================================================

def make_fewshot_split(test_indices, y, k_per_class=5, seed=42):
    rng = np.random.default_rng(seed)

    support_idx = []
    query_idx = []

    for c in range(N_CLASSES):
        cls_idx = test_indices[y[test_indices] == c]
        rng.shuffle(cls_idx)

        k = min(k_per_class, len(cls_idx) // 2)
        support_idx.extend(cls_idx[:k])
        query_idx.extend(cls_idx[k:])

        print(f"    class {c} ({ID_TO_LABEL[c]}): total={len(cls_idx)}, support={k}, query={len(cls_idx)-k}")

    support_idx = np.array(support_idx)
    query_idx = np.array(query_idx)

    return support_idx, query_idx

def standardize_raw_by_train(X_train, X_other):
    """
    Per-channel standardization using train subject stats.
    X shape: (N, T, C)
    """
    mean = X_train.mean(axis=(0,1), keepdims=True)
    std = X_train.std(axis=(0,1), keepdims=True) + 1e-6
    return (X_other - mean) / std

def standardize_features_by_train(F_train, F_other):
    mean = F_train.mean(axis=0, keepdims=True)
    std = F_train.std(axis=0, keepdims=True) + 1e-6
    return (F_other - mean) / std

In [ ]:
# ============================================================
# TRAINING: CNN ONLY
# ============================================================

def run_loso_cnn_only():
    print("\n" + "="*100)
    print("RUNNING BASELINE 1: CNN ONLY")
    print("="*100)

    fold_results = []
    unique_subjects = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#"*100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#"*100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx] == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED+fold)

        X_train_raw = X[train_idx]
        X_support_raw = X[support_idx]
        X_query_raw = X[query_idx]

        # standardize using train only
        train_mean = X_train_raw.mean(axis=(0,1), keepdims=True)
        train_std = X_train_raw.std(axis=(0,1), keepdims=True) + 1e-6

        X_train = (X_train_raw - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query = (X_query_raw - train_mean) / train_std

        y_train = y[train_idx]
        y_support = y[support_idx]
        y_query = y[query_idx]

        train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader = make_loader(X_query, y_query, batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNN(n_channels=8, n_classes=3, emb_dim=128).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = 0.0
        best_zs_acc = 0.0

        for epoch in range(1, EPOCHS+1):
            model.train()
            total_loss = 0.0
            all_preds, all_true = [], []

            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)
                preds = logits.argmax(dim=1)

                all_preds.extend(preds.detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(all_true, all_preds)

            zs_acc, zs_cm, _, _ = eval_classifier_cnn(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_cnn(model, support_loader)
            Z_q, y_q_t = get_embeddings_cnn(model, query_loader)
            fs_acc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc = max(best_fs_acc, fs_acc)
            best_zs_acc = max(best_zs_acc, zs_acc)

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
                f"zero_shot_query_acc={zs_acc:.4f} | fewshot_proto_acc={fs_acc:.4f} | "
                f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}"
            )

            print("  Zero-shot CM:")
            print(zs_cm)
            print("  Few-shot prototype CM:")
            print(fs_cm)

        print("\nFINAL FOLD RESULT")
        print(f"Subject {test_subj} | best zero-shot={best_zs_acc:.4f} | best few-shot={best_fs_acc:.4f}")

        fold_results.append({
            "subject": test_subj,
            "best_zero_shot": best_zs_acc,
            "best_fewshot": best_fs_acc,
            "n_train": len(train_idx),
            "n_support": len(support_idx),
            "n_query": len(query_idx),
        })

    results_df = pd.DataFrame(fold_results)
    print("\n" + "="*100)
    print("CNN ONLY LOSO SUMMARY")
    print("="*100)
    display(results_df)
    print("Mean zero-shot:", results_df["best_zero_shot"].mean())
    print("Std zero-shot :", results_df["best_zero_shot"].std())
    print("Mean few-shot :", results_df["best_fewshot"].mean())
    print("Std few-shot  :", results_df["best_fewshot"].std())

    out_path = os.path.join(SAVE_DIR, "baseline_cnn_only_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

In [ ]:
def run_loso_cnn_feature_fusion():
    print("\n" + "="*100)
    print("RUNNING BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION")
    print("="*100)

    fold_results = []
    unique_subjects = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#"*100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#"*100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx] == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED+fold)

        X_train_raw = X[train_idx]
        X_support_raw = X[support_idx]
        X_query_raw = X[query_idx]

        F_train_raw = F_all[train_idx]
        F_support_raw = F_all[support_idx]
        F_query_raw = F_all[query_idx]

        # raw standardization using train only
        train_mean = X_train_raw.mean(axis=(0,1), keepdims=True)
        train_std = X_train_raw.std(axis=(0,1), keepdims=True) + 1e-6

        X_train = (X_train_raw - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query = (X_query_raw - train_mean) / train_std

        # feature standardization using train only
        feat_mean = F_train_raw.mean(axis=0, keepdims=True)
        feat_std = F_train_raw.std(axis=0, keepdims=True) + 1e-6

        F_train = (F_train_raw - feat_mean) / feat_std
        F_support = (F_support_raw - feat_mean) / feat_std
        F_query = (F_query_raw - feat_mean) / feat_std

        y_train = y[train_idx]
        y_support = y[support_idx]
        y_query = y[query_idx]

        train_loader = make_loader(X_train, y_train, features=F_train, batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, features=F_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader = make_loader(X_query, y_query, features=F_query, batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNNFeatureFusion(
            n_channels=8,
            n_features=F_all.shape[1],
            n_classes=3,
            emb_dim=128,
            feat_dim=64
        ).to(device)

        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = 0.0
        best_zs_acc = 0.0

        for epoch in range(1, EPOCHS+1):
            model.train()
            total_loss = 0.0
            all_preds, all_true = [], []

            for xb, fb, yb in train_loader:
                xb = xb.to(device)
                fb = fb.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                logits, _ = model(xb, fb)
                loss = criterion(logits, yb)
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)
                preds = logits.argmax(dim=1)

                all_preds.extend(preds.detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(all_true, all_preds)

            zs_acc, zs_cm, _, _ = eval_classifier_fusion(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_fusion(model, support_loader)
            Z_q, y_q_t = get_embeddings_fusion(model, query_loader)
            fs_acc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc = max(best_fs_acc, fs_acc)
            best_zs_acc = max(best_zs_acc, zs_acc)

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
                f"zero_shot_query_acc={zs_acc:.4f} | fewshot_proto_acc={fs_acc:.4f} | "
                f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}"
            )

            print("  Zero-shot CM:")
            print(zs_cm)
            print("  Few-shot prototype CM:")
            print(fs_cm)

        print("\nFINAL FOLD RESULT")
        print(f"Subject {test_subj} | best zero-shot={best_zs_acc:.4f} | best few-shot={best_fs_acc:.4f}")

        fold_results.append({
            "subject": test_subj,
            "best_zero_shot": best_zs_acc,
            "best_fewshot": best_fs_acc,
            "n_train": len(train_idx),
            "n_support": len(support_idx),
            "n_query": len(query_idx),
        })

    results_df = pd.DataFrame(fold_results)
    print("\n" + "="*100)
    print("CNN + FEATURE FUSION LOSO SUMMARY")
    print("="*100)
    display(results_df)
    print("Mean zero-shot:", results_df["best_zero_shot"].mean())
    print("Std zero-shot :", results_df["best_zero_shot"].std())
    print("Mean few-shot :", results_df["best_fewshot"].mean())
    print("Std few-shot  :", results_df["best_fewshot"].std())

    out_path = os.path.join(SAVE_DIR, "baseline_cnn_feature_fusion_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df


In [ ]:
# ============================================================
# RUN BOTH BASELINES
# ============================================================

cnn_only_results = run_loso_cnn_only()



RUNNING BASELINE 1: CNN ONLY

####################################################################################################
FOLD 1/14 | HELD OUT SUBJECT = 1
####################################################################################################
Train windows: 812
Held-out windows: 67
Train label counts: {'lowlevel': 219, 'midlevel': 241, 'highlevel': 352}
Held label counts : {'lowlevel': 21, 'midlevel': 22, 'highlevel': 24}
    class 0 (lowlevel): total=21, support=5, query=16
    class 1 (midlevel): total=22, support=5, query=17
    class 2 (highlevel): total=24, support=5, query=19
Epoch 01/15 | loss=1.0982 | train_acc=0.3547 | zero_shot_query_acc=0.3077 | fewshot_proto_acc=0.4038 | best_zs=0.3077 | best_fs=0.4038
  Zero-shot CM:
[[16  0  0]
 [17  0  0]
 [18  1  0]]
  Few-shot prototype CM:
[[ 6  2  8]
 [ 4  0 13]
 [ 4  0 15]]
Epoch 02/15 | loss=1.0702 | train_acc=0.4052 | zero_shot_query_acc=0.2692 | fewshot_proto_acc=0.5192 | best_zs=0.3077 | best_fs=0.5192
  Z

,subject,best_zero_shot,best_fewshot,n_train,n_support,n_query
0,1,0.326923,0.557692,812,15,52
1,2,0.517857,0.571429,808,15,56
2,3,0.435897,0.692308,825,15,39
3,5,0.552632,0.500000,826,15,38
4,6,0.454545,0.545455,820,15,44
5,7,1.000000,1.000000,877,1,1
6,8,0.436170,0.382979,770,15,94
7,9,0.621622,0.486486,790,15,74
8,10,0.457143,0.571429,829,15,35
9,11,0.446429,0.464286,808,15,56


Mean zero-shot: 0.5286787381724192
Std zero-shot : 0.15532620180062945
Mean few-shot : 0.5719247187408281
Std few-shot  : 0.14786626406083112
Saved: /content/drive/MyDrive/eeg_qc_outputs/baseline_cnn_only_loso_results.csv


In [ ]:
fusion_results = run_loso_cnn_feature_fusion()


RUNNING BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION

####################################################################################################
FOLD 1/14 | HELD OUT SUBJECT = 1
####################################################################################################
Train windows: 812
Held-out windows: 67
Train label counts: {'lowlevel': 219, 'midlevel': 241, 'highlevel': 352}
Held label counts : {'lowlevel': 21, 'midlevel': 22, 'highlevel': 24}
    class 0 (lowlevel): total=21, support=5, query=16
    class 1 (midlevel): total=22, support=5, query=17
    class 2 (highlevel): total=24, support=5, query=19
Epoch 01/15 | loss=1.0938 | train_acc=0.3387 | zero_shot_query_acc=0.3462 | fewshot_proto_acc=0.3654 | best_zs=0.3462 | best_fs=0.3654
  Zero-shot CM:
[[ 6  4  6]
 [ 2 12  3]
 [ 7 12  0]]
  Few-shot prototype CM:
[[ 5 10  1]
 [ 3  7  7]
 [ 2 10  7]]
Epoch 02/15 | loss=1.0606 | train_acc=0.4298 | zero_shot_query_acc=0.3269 | fewshot_proto_acc=0.4615 | best_zs=0.3

,subject,best_zero_shot,best_fewshot,n_train,n_support,n_query
0,1,0.346154,0.500000,812,15,52
1,2,0.535714,0.589286,808,15,56
2,3,0.384615,0.666667,825,15,39
3,5,0.500000,0.552632,826,15,38
4,6,0.522727,0.522727,820,15,44
5,7,1.000000,1.000000,877,1,1
6,8,0.446809,0.372340,770,15,94
7,9,0.405405,0.648649,790,15,74
8,10,0.542857,0.457143,829,15,35
9,11,0.464286,0.482143,808,15,56


Mean zero-shot: 0.5029442198249189
Std zero-shot : 0.15497529195090187
Mean few-shot : 0.5627851986428614
Std few-shot  : 0.15296900414980655
Saved: /content/drive/MyDrive/eeg_qc_outputs/baseline_cnn_feature_fusion_loso_results.csv


In [ ]:
print("CNN-only results:")
display(cnn_only_results)


CNN-only results:


,subject,best_zero_shot,best_fewshot,n_train,n_support,n_query
0,1,0.326923,0.557692,812,15,52
1,2,0.517857,0.571429,808,15,56
2,3,0.435897,0.692308,825,15,39
3,5,0.552632,0.500000,826,15,38
4,6,0.454545,0.545455,820,15,44
5,7,1.000000,1.000000,877,1,1
6,8,0.436170,0.382979,770,15,94
7,9,0.621622,0.486486,790,15,74
8,10,0.457143,0.571429,829,15,35
9,11,0.446429,0.464286,808,15,56


In [ ]:
print("Fusion results:")
display(fusion_results)

Fusion results:


,subject,best_zero_shot,best_fewshot,n_train,n_support,n_query
0,1,0.346154,0.500000,812,15,52
1,2,0.535714,0.589286,808,15,56
2,3,0.384615,0.666667,825,15,39
3,5,0.500000,0.552632,826,15,38
4,6,0.522727,0.522727,820,15,44
5,7,1.000000,1.000000,877,1,1
6,8,0.446809,0.372340,770,15,94
7,9,0.405405,0.648649,790,15,74
8,10,0.542857,0.457143,829,15,35
9,11,0.464286,0.482143,808,15,56


In [ ]:
# ============================================================
# BALANCED LOSO EEG BASELINE
# Drops invalid subject 7
# Balances low/mid/high windows WITHIN EACH SUBJECT
# Runs CNN-only first, then CNN + handcrafted feature fusion
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, random
import numpy as np
import pandas as pd

from scipy.signal import welch
from scipy.stats import entropy

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score

# ============================================================
# CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

SAVE_DIR = "/content/drive/MyDrive/eeg_qc_outputs"
NPZ_PATH = os.path.join(SAVE_DIR, "clean_eeg_windows_qc.npz")

FS = 250
N_CLASSES = 3
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
FEW_SHOT_PER_CLASS = 5

DROP_SUBJECTS = [7]

LABEL_REMAP = {
    "lowlevel": 0,
    "midlevel": 1,
    "highlevel": 2
}

ID_TO_LABEL = {
    0: "lowlevel",
    1: "midlevel",
    2: "highlevel"
}

# ============================================================
# LOAD DATA
# ============================================================

data = np.load(NPZ_PATH, allow_pickle=True)

X_raw_all = data["X"].astype(np.float32)
labels_all = data["labels"]
subjects_all = data["subjects"]
tasks_all = data["tasks"]
file_paths_all = data["file_paths"]
start_secs_all = data["start_secs"]

print("Original X:", X_raw_all.shape)
print("Tasks:", np.unique(tasks_all))
print("Labels:", np.unique(labels_all))
print("Subjects:", np.unique(subjects_all))

# Arithmetic only, no natural, drop invalid subjects
mask = (
    (tasks_all == "Arithmetic") &
    np.isin(labels_all, ["lowlevel", "midlevel", "highlevel"]) &
    (~np.isin(subjects_all, DROP_SUBJECTS))
)

X_raw = X_raw_all[mask]
labels = labels_all[mask]
subjects = subjects_all[mask]
file_paths = file_paths_all[mask]
start_secs = start_secs_all[mask]

y = np.array([LABEL_REMAP[l] for l in labels])

print("\nAfter Arithmetic non-natural + drop subject 7:")
print("X:", X_raw.shape)
print("Subjects:", np.unique(subjects))
print("Label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

# ============================================================
# BALANCE WITHIN EACH SUBJECT
# ============================================================

def balance_within_subjects(X, y, subjects, labels, file_paths, start_secs, seed=42):
    rng = np.random.default_rng(seed)

    keep_indices = []

    print("\n" + "=" * 100)
    print("BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT")
    print("=" * 100)

    for s in sorted(np.unique(subjects)):
        subj_idx = np.where(subjects == s)[0]

        class_counts = {
            c: int(np.sum(y[subj_idx] == c))
            for c in range(N_CLASSES)
        }

        min_count = min(class_counts.values())

        print(f"Subject {s}: raw counts={class_counts} -> keeping {min_count}/class")

        if min_count == 0:
            print(f"  SKIPPING subject {s}: missing a class")
            continue

        for c in range(N_CLASSES):
            cls_idx = subj_idx[y[subj_idx] == c]
            chosen = rng.choice(cls_idx, size=min_count, replace=False)
            keep_indices.extend(chosen.tolist())

    keep_indices = np.array(sorted(keep_indices))

    return (
        X[keep_indices],
        y[keep_indices],
        subjects[keep_indices],
        labels[keep_indices],
        file_paths[keep_indices],
        start_secs[keep_indices],
        keep_indices
    )

X, y, subjects, labels, file_paths, start_secs, kept_original_indices = balance_within_subjects(
    X_raw, y, subjects, labels, file_paths, start_secs, seed=SEED
)

print("\nBalanced X:", X.shape)
print("Balanced global label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

balanced_summary = []
for s in sorted(np.unique(subjects)):
    row = {"subject": s}
    for c in range(N_CLASSES):
        row[ID_TO_LABEL[c]] = int(np.sum((subjects == s) & (y == c)))
    row["total"] = int(np.sum(subjects == s))
    balanced_summary.append(row)

balanced_summary = pd.DataFrame(balanced_summary)
display(balanced_summary)

# ============================================================
# HANDCRAFTED FEATURES
# ============================================================

bands = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 35),
}

def bandpower_from_psd(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return np.trapezoid(pxx[idx], f[idx])

def spectral_entropy_from_psd(pxx):
    p = pxx / (np.sum(pxx) + 1e-12)
    return entropy(p)

def extract_features_one_window(w, fs=250):
    feats = []

    ch_mean = np.mean(w, axis=0)
    ch_std = np.std(w, axis=0)
    ch_var = np.var(w, axis=0)
    ch_ptp = np.ptp(w, axis=0)

    feats.extend(ch_mean.tolist())
    feats.extend(ch_std.tolist())
    feats.extend(ch_var.tolist())
    feats.extend(ch_ptp.tolist())

    feats.append(np.mean(ch_var))
    feats.append(np.max(ch_var))
    feats.append(np.mean(ch_ptp))
    feats.append(np.max(ch_ptp))

    band_means = {b: [] for b in bands}
    spec_ents = []

    for ch in range(w.shape[1]):
        f, pxx = welch(w[:, ch], fs=fs, nperseg=512)

        for name, (lo, hi) in bands.items():
            bp = bandpower_from_psd(f, pxx, lo, hi)
            band_means[name].append(bp)

        spec_ents.append(spectral_entropy_from_psd(pxx))

    for name in ["theta", "alpha", "beta", "gamma"]:
        vals = np.array(band_means[name])
        feats.append(np.mean(vals))
        feats.append(np.log(np.mean(vals) + 1e-12))
        feats.append(np.std(vals))

    theta = np.mean(band_means["theta"])
    alpha = np.mean(band_means["alpha"])
    beta = np.mean(band_means["beta"])
    gamma = np.mean(band_means["gamma"])

    feats.append(theta / (alpha + 1e-12))
    feats.append(theta / (beta + 1e-12))
    feats.append(alpha / (beta + 1e-12))
    feats.append(gamma / (beta + 1e-12))

    feats.append(np.log(theta + 1e-12) - np.log(alpha + 1e-12))
    feats.append(np.log(theta + 1e-12) - np.log(beta + 1e-12))
    feats.append(np.log(alpha + 1e-12) - np.log(beta + 1e-12))

    feats.append(np.mean(spec_ents))
    feats.append(np.std(spec_ents))

    return np.array(feats, dtype=np.float32)

FEATURE_CACHE = os.path.join(SAVE_DIR, "arithmetic_balanced_baseline_features.npy")

if os.path.exists(FEATURE_CACHE):
    print("\nLoading cached balanced handcrafted features...")
    F_all = np.load(FEATURE_CACHE)
    if len(F_all) != len(X):
        print("Cached feature size mismatch. Recomputing.")
        os.remove(FEATURE_CACHE)
        F_all = None
else:
    F_all = None

if F_all is None:
    print("\nExtracting handcrafted features for BALANCED dataset...")
    feats = []
    for i in range(len(X)):
        if i % 100 == 0:
            print(f"  extracting {i}/{len(X)}")
        feats.append(extract_features_one_window(X[i], FS))
    F_all = np.stack(feats).astype(np.float32)
    np.save(FEATURE_CACHE, F_all)
    print("Saved features:", FEATURE_CACHE)

print("Feature matrix:", F_all.shape)

# ============================================================
# DATASET
# ============================================================

class EEGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, features=None):
        self.X = X
        self.y = y.astype(np.int64)
        self.features = features

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].T, dtype=torch.float32)
        label = torch.tensor(self.y[idx], dtype=torch.long)

        if self.features is None:
            return x, label
        else:
            f = torch.tensor(self.features[idx], dtype=torch.float32)
            return x, f, label

def make_loader(X, y, features=None, batch_size=64, shuffle=False, weighted=False):
    ds = EEGDataset(X, y, features)

    if weighted:
        class_counts = np.bincount(y, minlength=N_CLASSES)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[y]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=False)

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

# ============================================================
# MODELS
# ============================================================

class EEGCNN(nn.Module):
    def __init__(self, n_channels=8, n_classes=3, emb_dim=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.proj = nn.Sequential(
            nn.Linear(128, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x):
        h = self.encoder(x)
        h = self.pool(h).squeeze(-1)
        z = self.proj(h)
        return z

    def forward(self, x):
        z = self.forward_features(x)
        logits = self.classifier(z)
        return logits, z


class EEGCNNFeatureFusion(nn.Module):
    def __init__(self, n_channels=8, n_features=1, n_classes=3, emb_dim=128, feat_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.raw_proj = nn.Sequential(
            nn.Linear(128, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.feat_proj = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, feat_dim),
            nn.GELU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(emb_dim + feat_dim, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x, features):
        h = self.encoder(x)
        h = self.pool(h).squeeze(-1)

        z_raw = self.raw_proj(h)
        z_feat = self.feat_proj(features)

        z = torch.cat([z_raw, z_feat], dim=1)
        z = self.fusion(z)
        return z

    def forward(self, x, features):
        z = self.forward_features(x, features)
        logits = self.classifier(z)
        return logits, z

# ============================================================
# EVAL HELPERS
# ============================================================

@torch.no_grad()
def eval_classifier_cnn(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for x, yb in loader:
        x = x.to(device)
        yb = yb.to(device)

        logits, _ = model(x)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds, labels=[0,1,2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def eval_classifier_fusion(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for x, f, yb in loader:
        x = x.to(device)
        f = f.to(device)
        yb = yb.to(device)

        logits, _ = model(x, f)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds, labels=[0,1,2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def get_embeddings_cnn(model, loader):
    model.eval()
    Z, Y = [], []

    for x, yb in loader:
        x = x.to(device)
        _, z = model(x)
        Z.append(z.cpu())
        Y.append(yb)

    return torch.cat(Z, dim=0), torch.cat(Y, dim=0)

@torch.no_grad()
def get_embeddings_fusion(model, loader):
    model.eval()
    Z, Y = [], []

    for x, f, yb in loader:
        x = x.to(device)
        f = f.to(device)
        _, z = model(x, f)
        Z.append(z.cpu())
        Y.append(yb)

    return torch.cat(Z, dim=0), torch.cat(Y, dim=0)

def prototype_eval(Z_support, y_support, Z_query, y_query):
    Z_support = F.normalize(Z_support, dim=1)
    Z_query = F.normalize(Z_query, dim=1)

    prototypes = []

    for c in range(N_CLASSES):
        mask = (y_support == c)

        if mask.sum() == 0:
            proto = torch.zeros(Z_support.shape[1])
        else:
            proto = Z_support[mask].mean(dim=0)
            proto = F.normalize(proto, dim=0)

        prototypes.append(proto)

    prototypes = torch.stack(prototypes, dim=0)

    sims = Z_query @ prototypes.T
    preds = sims.argmax(dim=1)

    acc = accuracy_score(y_query.numpy(), preds.numpy())
    bacc = balanced_accuracy_score(y_query.numpy(), preds.numpy())
    cm = confusion_matrix(y_query.numpy(), preds.numpy(), labels=[0,1,2])

    return acc, bacc, cm, preds.numpy()

# ============================================================
# SPLITTING
# ============================================================

def make_fewshot_split(test_indices, y, k_per_class=5, seed=42):
    rng = np.random.default_rng(seed)

    support_idx = []
    query_idx = []

    print("  Few-shot split:")

    for c in range(N_CLASSES):
        cls_idx = test_indices[y[test_indices] == c]
        rng.shuffle(cls_idx)

        k = min(k_per_class, len(cls_idx) // 2)

        support_idx.extend(cls_idx[:k])
        query_idx.extend(cls_idx[k:])

        percent = 100 * k / len(cls_idx) if len(cls_idx) > 0 else 0

        print(
            f"    class {c} ({ID_TO_LABEL[c]}): "
            f"total={len(cls_idx)}, support={k}, query={len(cls_idx)-k}, support%={percent:.1f}%"
        )

    return np.array(support_idx), np.array(query_idx)

# ============================================================
# BASELINE MAJORITY CLASS
# ============================================================

def majority_baseline(y_query):
    vals, counts = np.unique(y_query, return_counts=True)
    majority = vals[np.argmax(counts)]
    preds = np.full_like(y_query, fill_value=majority)

    acc = accuracy_score(y_query, preds)
    bacc = balanced_accuracy_score(y_query, preds)
    cm = confusion_matrix(y_query, preds, labels=[0,1,2])

    return acc, bacc, cm

# ============================================================
# CNN ONLY LOSO
# ============================================================

def run_loso_cnn_only():
    print("\n" + "="*100)
    print("RUNNING BALANCED BASELINE 1: CNN ONLY")
    print("="*100)

    fold_results = []
    unique_subjects = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#"*100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#"*100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx] == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED+fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query")
            continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)

        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        X_train_raw = X[train_idx]
        X_support_raw = X[support_idx]
        X_query_raw = X[query_idx]

        train_mean = X_train_raw.mean(axis=(0,1), keepdims=True)
        train_std = X_train_raw.std(axis=(0,1), keepdims=True) + 1e-6

        X_train = (X_train_raw - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query = (X_query_raw - train_mean) / train_std

        y_train = y[train_idx]
        y_support = y[support_idx]
        y_query = y[query_idx]

        train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader = make_loader(X_query, y_query, batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNN(n_channels=8, n_classes=3, emb_dim=128).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = 0.0
        best_fs_bacc = 0.0
        best_zs_acc = 0.0
        best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS+1):
            model.train()
            total_loss = 0.0
            all_preds, all_true = [], []

            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)

                preds = logits.argmax(dim=1)
                all_preds.extend(preds.detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(all_true, all_preds)
            train_bacc = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_cnn(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_cnn(model, support_loader)
            Z_q, y_q_t = get_embeddings_cnn(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc = max(best_fs_acc, fs_acc)
            best_fs_bacc = max(best_fs_bacc, fs_bacc)
            best_zs_acc = max(best_zs_acc, zs_acc)
            best_zs_bacc = max(best_zs_bacc, zs_bacc)

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.4f} | "
                f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                f"zero_shot_acc={zs_acc:.4f} | zero_shot_bacc={zs_bacc:.4f} | "
                f"fewshot_acc={fs_acc:.4f} | fewshot_bacc={fs_bacc:.4f} | "
                f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}"
            )

            print("  Zero-shot CM:")
            print(zs_cm)
            print("  Few-shot prototype CM:")
            print(fs_cm)

        print("\nFINAL FOLD RESULT")
        print(
            f"Subject {test_subj} | "
            f"best zero-shot={best_zs_acc:.4f} | best zero-shot bacc={best_zs_bacc:.4f} | "
            f"best few-shot={best_fs_acc:.4f} | best few-shot bacc={best_fs_bacc:.4f}"
        )

        fold_results.append({
            "subject": test_subj,
            "best_zero_shot": best_zs_acc,
            "best_zero_shot_bacc": best_zs_bacc,
            "best_fewshot": best_fs_acc,
            "best_fewshot_bacc": best_fs_bacc,
            "majority_acc": maj_acc,
            "majority_bacc": maj_bacc,
            "n_train": len(train_idx),
            "n_support": len(support_idx),
            "n_query": len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx)
        })

    results_df = pd.DataFrame(fold_results)

    print("\n" + "="*100)
    print("BALANCED CNN ONLY LOSO SUMMARY")
    print("="*100)
    display(results_df)

    print("Mean zero-shot:", results_df["best_zero_shot"].mean())
    print("Mean zero-shot bacc:", results_df["best_zero_shot_bacc"].mean())
    print("Mean few-shot:", results_df["best_fewshot"].mean())
    print("Mean few-shot bacc:", results_df["best_fewshot_bacc"].mean())
    print("Mean majority acc:", results_df["majority_acc"].mean())
    print("Mean support %:", results_df["support_percent_total"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_only_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# CNN + FEATURE FUSION LOSO
# ============================================================

def run_loso_cnn_feature_fusion():
    print("\n" + "="*100)
    print("RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION")
    print("="*100)

    fold_results = []
    unique_subjects = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#"*100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#"*100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx] == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED+fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query")
            continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)

        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        X_train_raw = X[train_idx]
        X_support_raw = X[support_idx]
        X_query_raw = X[query_idx]

        F_train_raw = F_all[train_idx]
        F_support_raw = F_all[support_idx]
        F_query_raw = F_all[query_idx]

        train_mean = X_train_raw.mean(axis=(0,1), keepdims=True)
        train_std = X_train_raw.std(axis=(0,1), keepdims=True) + 1e-6

        X_train = (X_train_raw - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query = (X_query_raw - train_mean) / train_std

        feat_mean = F_train_raw.mean(axis=0, keepdims=True)
        feat_std = F_train_raw.std(axis=0, keepdims=True) + 1e-6

        F_train = (F_train_raw - feat_mean) / feat_std
        F_support = (F_support_raw - feat_mean) / feat_std
        F_query = (F_query_raw - feat_mean) / feat_std

        y_train = y[train_idx]
        y_support = y[support_idx]
        y_query = y[query_idx]

        train_loader = make_loader(X_train, y_train, features=F_train, batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, features=F_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader = make_loader(X_query, y_query, features=F_query, batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNNFeatureFusion(
            n_channels=8,
            n_features=F_all.shape[1],
            n_classes=3,
            emb_dim=128,
            feat_dim=64
        ).to(device)

        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = 0.0
        best_fs_bacc = 0.0
        best_zs_acc = 0.0
        best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS+1):
            model.train()
            total_loss = 0.0
            all_preds, all_true = [], []

            for xb, fb, yb in train_loader:
                xb = xb.to(device)
                fb = fb.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                logits, _ = model(xb, fb)
                loss = criterion(logits, yb)
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)

                preds = logits.argmax(dim=1)
                all_preds.extend(preds.detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(all_true, all_preds)
            train_bacc = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_fusion(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_fusion(model, support_loader)
            Z_q, y_q_t = get_embeddings_fusion(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc = max(best_fs_acc, fs_acc)
            best_fs_bacc = max(best_fs_bacc, fs_bacc)
            best_zs_acc = max(best_zs_acc, zs_acc)
            best_zs_bacc = max(best_zs_bacc, zs_bacc)

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.4f} | "
                f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                f"zero_shot_acc={zs_acc:.4f} | zero_shot_bacc={zs_bacc:.4f} | "
                f"fewshot_acc={fs_acc:.4f} | fewshot_bacc={fs_bacc:.4f} | "
                f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}"
            )

            print("  Zero-shot CM:")
            print(zs_cm)
            print("  Few-shot prototype CM:")
            print(fs_cm)

        print("\nFINAL FOLD RESULT")
        print(
            f"Subject {test_subj} | "
            f"best zero-shot={best_zs_acc:.4f} | best zero-shot bacc={best_zs_bacc:.4f} | "
            f"best few-shot={best_fs_acc:.4f} | best few-shot bacc={best_fs_bacc:.4f}"
        )

        fold_results.append({
            "subject": test_subj,
            "best_zero_shot": best_zs_acc,
            "best_zero_shot_bacc": best_zs_bacc,
            "best_fewshot": best_fs_acc,
            "best_fewshot_bacc": best_fs_bacc,
            "majority_acc": maj_acc,
            "majority_bacc": maj_bacc,
            "n_train": len(train_idx),
            "n_support": len(support_idx),
            "n_query": len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx)
        })

    results_df = pd.DataFrame(fold_results)

    print("\n" + "="*100)
    print("BALANCED CNN + FEATURE FUSION LOSO SUMMARY")
    print("="*100)
    display(results_df)

    print("Mean zero-shot:", results_df["best_zero_shot"].mean())
    print("Mean zero-shot bacc:", results_df["best_zero_shot_bacc"].mean())
    print("Mean few-shot:", results_df["best_fewshot"].mean())
    print("Mean few-shot bacc:", results_df["best_fewshot_bacc"].mean())
    print("Mean majority acc:", results_df["majority_acc"].mean())
    print("Mean support %:", results_df["support_percent_total"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_feature_fusion_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# RUN
# ============================================================

cnn_only_results = run_loso_cnn_only()


Mounted at /content/drive
DEVICE: cuda
Original X: (2371, 1000, 8)
Tasks: ['Arithmetic' 'Stroop']
Labels: ['highlevel' 'lowlevel' 'midlevel' 'natural']
Subjects: [ 1  2  3  5  6  7  8  9 10 11 12 13 14 15]

After Arithmetic non-natural + drop subject 7:
X: (877, 1000, 8)
Subjects: [ 1  2  3  5  6  8  9 10 11 12 13 14 15]
Label counts: {'lowlevel': 240, 'midlevel': 261, 'highlevel': 376}

BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT
Subject 1: raw counts={0: 21, 1: 22, 2: 24} -> keeping 21/class
Subject 2: raw counts={0: 18, 1: 19, 2: 34} -> keeping 18/class
Subject 3: raw counts={0: 13, 1: 19, 2: 22} -> keeping 13/class
Subject 5: raw counts={0: 13, 1: 20, 2: 20} -> keeping 13/class
Subject 6: raw counts={0: 16, 1: 18, 2: 25} -> keeping 16/class
Subject 8: raw counts={0: 35, 1: 34, 2: 40} -> keeping 34/class
Subject 9: raw counts={0: 17, 1: 17, 2: 55} -> keeping 17/class
Subject 10: raw counts={0: 14, 1: 16, 2: 20} -> keeping 14/class
Subject 11: raw counts={0: 21, 1: 22, 2: 28} -> ke

,subject,lowlevel,midlevel,highlevel,total
0,1,21,21,21,63
1,2,18,18,18,54
2,3,13,13,13,39
3,5,13,13,13,39
4,6,16,16,16,48
5,8,34,34,34,102
6,9,17,17,17,51
7,10,14,14,14,42
8,11,21,21,21,63
9,12,16,16,16,48



Extracting handcrafted features for BALANCED dataset...
  extracting 0/702
  extracting 100/702
  extracting 200/702
  extracting 300/702
  extracting 400/702
  extracting 500/702
  extracting 600/702
  extracting 700/702
Saved features: /content/drive/MyDrive/eeg_qc_outputs/arithmetic_balanced_baseline_features.npy
Feature matrix: (702, 57)

RUNNING BALANCED BASELINE 1: CNN ONLY

####################################################################################################
FOLD 1/13 | HELD OUT SUBJECT = 1
####################################################################################################
Train windows: 639
Held-out windows: 63
Train label counts: {'lowlevel': 213, 'midlevel': 213, 'highlevel': 213}
Held label counts : {'lowlevel': 21, 'midlevel': 21, 'highlevel': 21}
  Few-shot split:
    class 0 (lowlevel): total=21, support=5, query=16, support%=23.8%
    class 1 (midlevel): total=21, support=5, query=16, support%=23.8%
    class 2 (highlevel): total=21, supp

,subject,best_zero_shot,best_zero_shot_bacc,best_fewshot,best_fewshot_bacc,majority_acc,majority_bacc,n_train,n_support,n_query,support_percent_total
0,1,0.333333,0.333333,0.541667,0.541667,0.333333,0.333333,639,15,48,23.809524
1,2,0.589744,0.589744,0.564103,0.564103,0.333333,0.333333,648,15,39,27.777778
2,3,0.500000,0.500000,0.833333,0.833333,0.333333,0.333333,663,15,24,38.461538
3,5,0.583333,0.583333,0.541667,0.541667,0.333333,0.333333,663,15,24,38.461538
4,6,0.454545,0.454545,0.424242,0.424242,0.333333,0.333333,654,15,33,31.250000
5,8,0.367816,0.367816,0.390805,0.390805,0.333333,0.333333,600,15,87,14.705882
6,9,0.472222,0.472222,0.555556,0.555556,0.333333,0.333333,651,15,36,29.411765
7,10,0.444444,0.444444,0.481481,0.481481,0.333333,0.333333,660,15,27,35.714286
8,11,0.479167,0.479167,0.458333,0.458333,0.333333,0.333333,639,15,48,23.809524
9,12,0.484848,0.484848,0.454545,0.454545,0.333333,0.333333,654,15,33,31.250000


Mean zero-shot: 0.4608337617289871
Mean zero-shot bacc: 0.4608337617289872
Mean few-shot: 0.5192354678428948
Mean few-shot bacc: 0.5192354678428948
Mean majority acc: 0.3333333333333333
Mean support %: 29.497509582053517
Saved: /content/drive/MyDrive/eeg_qc_outputs/balanced_baseline_cnn_only_loso_results.csv


In [ ]:

fusion_results = run_loso_cnn_feature_fusion()




RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION

####################################################################################################
FOLD 1/13 | HELD OUT SUBJECT = 1
####################################################################################################
Train windows: 639
Held-out windows: 63
Train label counts: {'lowlevel': 213, 'midlevel': 213, 'highlevel': 213}
Held label counts : {'lowlevel': 21, 'midlevel': 21, 'highlevel': 21}
  Few-shot split:
    class 0 (lowlevel): total=21, support=5, query=16, support%=23.8%
    class 1 (midlevel): total=21, support=5, query=16, support%=23.8%
    class 2 (highlevel): total=21, support=5, query=16, support%=23.8%
Majority baseline query acc=0.3333 | balanced_acc=0.3333
[[16  0  0]
 [16  0  0]
 [16  0  0]]
Epoch 01/15 | loss=1.0945 | train_acc=0.3850 | train_bacc=0.3760 | zero_shot_acc=0.2500 | zero_shot_bacc=0.2500 | fewshot_acc=0.4375 | fewshot_bacc=0.4375 | best_zs=0.2500 | best_fs=0.4375
  Zero

,subject,best_zero_shot,best_zero_shot_bacc,best_fewshot,best_fewshot_bacc,majority_acc,majority_bacc,n_train,n_support,n_query,support_percent_total
0,1,0.354167,0.354167,0.541667,0.541667,0.333333,0.333333,639,15,48,23.809524
1,2,0.461538,0.461538,0.538462,0.538462,0.333333,0.333333,648,15,39,27.777778
2,3,0.541667,0.541667,0.791667,0.791667,0.333333,0.333333,663,15,24,38.461538
3,5,0.541667,0.541667,0.541667,0.541667,0.333333,0.333333,663,15,24,38.461538
4,6,0.545455,0.545455,0.454545,0.454545,0.333333,0.333333,654,15,33,31.250000
5,8,0.344828,0.344828,0.436782,0.436782,0.333333,0.333333,600,15,87,14.705882
6,9,0.361111,0.361111,0.583333,0.583333,0.333333,0.333333,651,15,36,29.411765
7,10,0.481481,0.481481,0.592593,0.592593,0.333333,0.333333,660,15,27,35.714286
8,11,0.395833,0.395833,0.437500,0.437500,0.333333,0.333333,639,15,48,23.809524
9,12,0.454545,0.454545,0.545455,0.545455,0.333333,0.333333,654,15,33,31.250000


Mean zero-shot: 0.43986328535400143
Mean zero-shot bacc: 0.43986328535400143
Mean few-shot: 0.556978873126088
Mean few-shot bacc: 0.556978873126088
Mean majority acc: 0.3333333333333333
Mean support %: 29.497509582053517
Saved: /content/drive/MyDrive/eeg_qc_outputs/balanced_baseline_cnn_feature_fusion_loso_results.csv


In [ ]:
print("\nDONE.")
print("CNN-only balanced results:")
display(cnn_only_results)



DONE.
CNN-only balanced results:


,subject,best_zero_shot,best_zero_shot_bacc,best_fewshot,best_fewshot_bacc,majority_acc,majority_bacc,n_train,n_support,n_query,support_percent_total
0,1,0.333333,0.333333,0.541667,0.541667,0.333333,0.333333,639,15,48,23.809524
1,2,0.589744,0.589744,0.564103,0.564103,0.333333,0.333333,648,15,39,27.777778
2,3,0.500000,0.500000,0.833333,0.833333,0.333333,0.333333,663,15,24,38.461538
3,5,0.583333,0.583333,0.541667,0.541667,0.333333,0.333333,663,15,24,38.461538
4,6,0.454545,0.454545,0.424242,0.424242,0.333333,0.333333,654,15,33,31.250000
5,8,0.367816,0.367816,0.390805,0.390805,0.333333,0.333333,600,15,87,14.705882
6,9,0.472222,0.472222,0.555556,0.555556,0.333333,0.333333,651,15,36,29.411765
7,10,0.444444,0.444444,0.481481,0.481481,0.333333,0.333333,660,15,27,35.714286
8,11,0.479167,0.479167,0.458333,0.458333,0.333333,0.333333,639,15,48,23.809524
9,12,0.484848,0.484848,0.454545,0.454545,0.333333,0.333333,654,15,33,31.250000


In [ ]:

print("Fusion balanced results:")
display(fusion_results)

Fusion balanced results:


,subject,best_zero_shot,best_zero_shot_bacc,best_fewshot,best_fewshot_bacc,majority_acc,majority_bacc,n_train,n_support,n_query,support_percent_total
0,1,0.354167,0.354167,0.541667,0.541667,0.333333,0.333333,639,15,48,23.809524
1,2,0.461538,0.461538,0.538462,0.538462,0.333333,0.333333,648,15,39,27.777778
2,3,0.541667,0.541667,0.791667,0.791667,0.333333,0.333333,663,15,24,38.461538
3,5,0.541667,0.541667,0.541667,0.541667,0.333333,0.333333,663,15,24,38.461538
4,6,0.545455,0.545455,0.454545,0.454545,0.333333,0.333333,654,15,33,31.250000
5,8,0.344828,0.344828,0.436782,0.436782,0.333333,0.333333,600,15,87,14.705882
6,9,0.361111,0.361111,0.583333,0.583333,0.333333,0.333333,651,15,36,29.411765
7,10,0.481481,0.481481,0.592593,0.592593,0.333333,0.333333,660,15,27,35.714286
8,11,0.395833,0.395833,0.437500,0.437500,0.333333,0.333333,639,15,48,23.809524
9,12,0.454545,0.454545,0.545455,0.545455,0.333333,0.333333,654,15,33,31.250000


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


ValueError: Mountpoint must not already contain files

In [ ]:
#Rerunning the same exact loop as before so I can visualize embeddings per subject

In [ ]:
# ============================================================
# BALANCED LOSO EEG BASELINE
# Drops invalid subject 7
# Balances low/mid/high windows WITHIN EACH SUBJECT
# Runs CNN-only first, then CNN + handcrafted feature fusion
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, random
import numpy as np
import pandas as pd

from scipy.signal import welch
from scipy.stats import entropy

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score

# ============================================================
# CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

SAVE_DIR = "/content/drive/MyDrive/eeg_qc_outputs"
NPZ_PATH = os.path.join(SAVE_DIR, "clean_eeg_windows_qc.npz")

FS = 250
N_CLASSES = 3
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
FEW_SHOT_PER_CLASS = 5

DROP_SUBJECTS = [7]

LABEL_REMAP = {
    "lowlevel": 0,
    "midlevel": 1,
    "highlevel": 2
}

ID_TO_LABEL = {
    0: "lowlevel",
    1: "midlevel",
    2: "highlevel"
}

# ============================================================
# LOAD DATA
# ============================================================

data = np.load(NPZ_PATH, allow_pickle=True)

X_raw_all = data["X"].astype(np.float32)
labels_all = data["labels"]
subjects_all = data["subjects"]
tasks_all = data["tasks"]
file_paths_all = data["file_paths"]
start_secs_all = data["start_secs"]

print("Original X:", X_raw_all.shape)
print("Tasks:", np.unique(tasks_all))
print("Labels:", np.unique(labels_all))
print("Subjects:", np.unique(subjects_all))

# Arithmetic only, no natural, drop invalid subjects
mask = (
    (tasks_all == "Arithmetic") &
    np.isin(labels_all, ["lowlevel", "midlevel", "highlevel"]) &
    (~np.isin(subjects_all, DROP_SUBJECTS))
)

X_raw = X_raw_all[mask]
labels = labels_all[mask]
subjects = subjects_all[mask]
file_paths = file_paths_all[mask]
start_secs = start_secs_all[mask]

y = np.array([LABEL_REMAP[l] for l in labels])

print("\nAfter Arithmetic non-natural + drop subject 7:")
print("X:", X_raw.shape)
print("Subjects:", np.unique(subjects))
print("Label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

# ============================================================
# BALANCE WITHIN EACH SUBJECT
# ============================================================

def balance_within_subjects(X, y, subjects, labels, file_paths, start_secs, seed=42):
    rng = np.random.default_rng(seed)

    keep_indices = []

    print("\n" + "=" * 100)
    print("BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT")
    print("=" * 100)

    for s in sorted(np.unique(subjects)):
        subj_idx = np.where(subjects == s)[0]

        class_counts = {
            c: int(np.sum(y[subj_idx] == c))
            for c in range(N_CLASSES)
        }

        min_count = min(class_counts.values())

        print(f"Subject {s}: raw counts={class_counts} -> keeping {min_count}/class")

        if min_count == 0:
            print(f"  SKIPPING subject {s}: missing a class")
            continue

        for c in range(N_CLASSES):
            cls_idx = subj_idx[y[subj_idx] == c]
            chosen = rng.choice(cls_idx, size=min_count, replace=False)
            keep_indices.extend(chosen.tolist())

    keep_indices = np.array(sorted(keep_indices))

    return (
        X[keep_indices],
        y[keep_indices],
        subjects[keep_indices],
        labels[keep_indices],
        file_paths[keep_indices],
        start_secs[keep_indices],
        keep_indices
    )

X, y, subjects, labels, file_paths, start_secs, kept_original_indices = balance_within_subjects(
    X_raw, y, subjects, labels, file_paths, start_secs, seed=SEED
)

print("\nBalanced X:", X.shape)
print("Balanced global label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

balanced_summary = []
for s in sorted(np.unique(subjects)):
    row = {"subject": s}
    for c in range(N_CLASSES):
        row[ID_TO_LABEL[c]] = int(np.sum((subjects == s) & (y == c)))
    row["total"] = int(np.sum(subjects == s))
    balanced_summary.append(row)

balanced_summary = pd.DataFrame(balanced_summary)
display(balanced_summary)

# ============================================================
# HANDCRAFTED FEATURES
# ============================================================

bands = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 35),
}

def bandpower_from_psd(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return np.trapezoid(pxx[idx], f[idx])

def spectral_entropy_from_psd(pxx):
    p = pxx / (np.sum(pxx) + 1e-12)
    return entropy(p)

def extract_features_one_window(w, fs=250):
    feats = []

    ch_mean = np.mean(w, axis=0)
    ch_std = np.std(w, axis=0)
    ch_var = np.var(w, axis=0)
    ch_ptp = np.ptp(w, axis=0)

    feats.extend(ch_mean.tolist())
    feats.extend(ch_std.tolist())
    feats.extend(ch_var.tolist())
    feats.extend(ch_ptp.tolist())

    feats.append(np.mean(ch_var))
    feats.append(np.max(ch_var))
    feats.append(np.mean(ch_ptp))
    feats.append(np.max(ch_ptp))

    band_means = {b: [] for b in bands}
    spec_ents = []

    for ch in range(w.shape[1]):
        f, pxx = welch(w[:, ch], fs=fs, nperseg=512)

        for name, (lo, hi) in bands.items():
            bp = bandpower_from_psd(f, pxx, lo, hi)
            band_means[name].append(bp)

        spec_ents.append(spectral_entropy_from_psd(pxx))

    for name in ["theta", "alpha", "beta", "gamma"]:
        vals = np.array(band_means[name])
        feats.append(np.mean(vals))
        feats.append(np.log(np.mean(vals) + 1e-12))
        feats.append(np.std(vals))

    theta = np.mean(band_means["theta"])
    alpha = np.mean(band_means["alpha"])
    beta = np.mean(band_means["beta"])
    gamma = np.mean(band_means["gamma"])

    feats.append(theta / (alpha + 1e-12))
    feats.append(theta / (beta + 1e-12))
    feats.append(alpha / (beta + 1e-12))
    feats.append(gamma / (beta + 1e-12))

    feats.append(np.log(theta + 1e-12) - np.log(alpha + 1e-12))
    feats.append(np.log(theta + 1e-12) - np.log(beta + 1e-12))
    feats.append(np.log(alpha + 1e-12) - np.log(beta + 1e-12))

    feats.append(np.mean(spec_ents))
    feats.append(np.std(spec_ents))

    return np.array(feats, dtype=np.float32)

FEATURE_CACHE = os.path.join(SAVE_DIR, "arithmetic_balanced_baseline_features.npy")

if os.path.exists(FEATURE_CACHE):
    print("\nLoading cached balanced handcrafted features...")
    F_all = np.load(FEATURE_CACHE)
    if len(F_all) != len(X):
        print("Cached feature size mismatch. Recomputing.")
        os.remove(FEATURE_CACHE)
        F_all = None
else:
    F_all = None

if F_all is None:
    print("\nExtracting handcrafted features for BALANCED dataset...")
    feats = []
    for i in range(len(X)):
        if i % 100 == 0:
            print(f"  extracting {i}/{len(X)}")
        feats.append(extract_features_one_window(X[i], FS))
    F_all = np.stack(feats).astype(np.float32)
    np.save(FEATURE_CACHE, F_all)
    print("Saved features:", FEATURE_CACHE)

print("Feature matrix:", F_all.shape)

# ============================================================
# DATASET
# ============================================================

class EEGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, features=None):
        self.X = X
        self.y = y.astype(np.int64)
        self.features = features

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].T, dtype=torch.float32)
        label = torch.tensor(self.y[idx], dtype=torch.long)

        if self.features is None:
            return x, label
        else:
            f = torch.tensor(self.features[idx], dtype=torch.float32)
            return x, f, label

def make_loader(X, y, features=None, batch_size=64, shuffle=False, weighted=False):
    ds = EEGDataset(X, y, features)

    if weighted:
        class_counts = np.bincount(y, minlength=N_CLASSES)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[y]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=False)

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

# ============================================================
# MODELS
# ============================================================

class EEGCNN(nn.Module):
    def __init__(self, n_channels=8, n_classes=3, emb_dim=128):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.proj = nn.Sequential(
            nn.Linear(128, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x):
        h = self.encoder(x)
        h = self.pool(h).squeeze(-1)
        z = self.proj(h)
        return z

    def forward(self, x):
        z = self.forward_features(x)
        logits = self.classifier(z)
        return logits, z


class EEGCNNFeatureFusion(nn.Module):
    def __init__(self, n_channels=8, n_features=1, n_classes=3, emb_dim=128, feat_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=15, padding=7),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.GELU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.raw_proj = nn.Sequential(
            nn.Linear(128, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.feat_proj = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, feat_dim),
            nn.GELU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(emb_dim + feat_dim, emb_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x, features):
        h = self.encoder(x)
        h = self.pool(h).squeeze(-1)

        z_raw = self.raw_proj(h)
        z_feat = self.feat_proj(features)

        z = torch.cat([z_raw, z_feat], dim=1)
        z = self.fusion(z)
        return z

    def forward(self, x, features):
        z = self.forward_features(x, features)
        logits = self.classifier(z)
        return logits, z

# ============================================================
# EVAL HELPERS
# ============================================================

@torch.no_grad()
def eval_classifier_cnn(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for x, yb in loader:
        x = x.to(device)
        yb = yb.to(device)

        logits, _ = model(x)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds, labels=[0,1,2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def eval_classifier_fusion(model, loader):
    model.eval()
    all_preds, all_true = [], []

    for x, f, yb in loader:
        x = x.to(device)
        f = f.to(device)
        yb = yb.to(device)

        logits, _ = model(x, f)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    acc = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm = confusion_matrix(all_true, all_preds, labels=[0,1,2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def get_embeddings_cnn(model, loader):
    model.eval()
    Z, Y = [], []

    for x, yb in loader:
        x = x.to(device)
        _, z = model(x)
        Z.append(z.cpu())
        Y.append(yb)

    return torch.cat(Z, dim=0), torch.cat(Y, dim=0)

@torch.no_grad()
def get_embeddings_fusion(model, loader):
    model.eval()
    Z, Y = [], []

    for x, f, yb in loader:
        x = x.to(device)
        f = f.to(device)
        _, z = model(x, f)
        Z.append(z.cpu())
        Y.append(yb)

    return torch.cat(Z, dim=0), torch.cat(Y, dim=0)

def prototype_eval(Z_support, y_support, Z_query, y_query):
    Z_support = F.normalize(Z_support, dim=1)
    Z_query = F.normalize(Z_query, dim=1)

    prototypes = []

    for c in range(N_CLASSES):
        mask = (y_support == c)

        if mask.sum() == 0:
            proto = torch.zeros(Z_support.shape[1])
        else:
            proto = Z_support[mask].mean(dim=0)
            proto = F.normalize(proto, dim=0)

        prototypes.append(proto)

    prototypes = torch.stack(prototypes, dim=0)

    sims = Z_query @ prototypes.T
    preds = sims.argmax(dim=1)

    acc = accuracy_score(y_query.numpy(), preds.numpy())
    bacc = balanced_accuracy_score(y_query.numpy(), preds.numpy())
    cm = confusion_matrix(y_query.numpy(), preds.numpy(), labels=[0,1,2])

    return acc, bacc, cm, preds.numpy()

# ============================================================
# EMBEDDING VISUALIZATION HELPERS
# Compatible with EEGCNN and EEGCNNFeatureFusion
# ============================================================

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

EMBED_PLOT_DIR = os.path.join(SAVE_DIR, "embedding_diagnostics")
os.makedirs(EMBED_PLOT_DIR, exist_ok=True)


@torch.no_grad()
def get_embeddings_any(model, loader, fusion=False):
    model.eval()
    Z, Y = [], []

    for batch in loader:
        if fusion:
            x, f, yb = batch
            x = x.to(device)
            f = f.to(device)
            _, z = model(x, f)
        else:
            x, yb = batch
            x = x.to(device)
            _, z = model(x)

        Z.append(z.detach().cpu().numpy())
        Y.append(yb.detach().cpu().numpy())

    Z = np.concatenate(Z, axis=0)
    Y = np.concatenate(Y, axis=0)

    return Z, Y


def plot_embeddings_pca_tsne(
    Z,
    Y,
    subject,
    model_name,
    epoch,
    split_name,
    save=True
):
    Z_scaled = StandardScaler().fit_transform(Z)

    # -------------------------
    # PCA
    # -------------------------
    pca = PCA(n_components=2, random_state=SEED)
    Z_pca = pca.fit_transform(Z_scaled)

    # -------------------------
    # t-SNE
    # -------------------------
    n = len(Z_scaled)
    perplexity = max(3, min(30, (n - 1) // 3))

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=SEED
    )

    Z_tsne = tsne.fit_transform(Z_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for c in range(N_CLASSES):
        idx = Y == c

        axes[0].scatter(
            Z_pca[idx, 0],
            Z_pca[idx, 1],
            s=45,
            alpha=0.75,
            label=ID_TO_LABEL[c]
        )

        axes[1].scatter(
            Z_tsne[idx, 0],
            Z_tsne[idx, 1],
            s=45,
            alpha=0.75,
            label=ID_TO_LABEL[c]
        )

    axes[0].set_title(
        f"{model_name} | Subject {subject} | Epoch {epoch} | {split_name}\n"
        f"PCA var={pca.explained_variance_ratio_.sum():.3f}"
    )
    axes[1].set_title(
        f"{model_name} | Subject {subject} | Epoch {epoch} | {split_name}\n"
        f"t-SNE perplexity={perplexity}"
    )

    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    axes[0].legend()
    axes[1].legend()

    plt.tight_layout()

    if save:
        fname = (
            f"{model_name}_subject_{subject}_epoch_{epoch:02d}_{split_name}"
            .replace(" ", "_")
            .replace("/", "_")
        )
        path = os.path.join(EMBED_PLOT_DIR, fname + ".png")
        plt.savefig(path, dpi=200)
        print("Saved embedding plot:", path)

    plt.show()


def plot_support_query_embeddings(
    Z_support,
    y_support,
    Z_query,
    y_query,
    subject,
    model_name,
    epoch,
    save=True
):
    Z_all = np.concatenate([Z_support, Z_query], axis=0)
    y_all = np.concatenate([y_support, y_query], axis=0)
    split = np.array(["support"] * len(Z_support) + ["query"] * len(Z_query))

    Z_scaled = StandardScaler().fit_transform(Z_all)

    pca = PCA(n_components=2, random_state=SEED)
    Z_pca = pca.fit_transform(Z_scaled)

    plt.figure(figsize=(7, 6))

    markers = {
        "support": "X",
        "query": "o"
    }

    for c in range(N_CLASSES):
        for sp in ["support", "query"]:
            idx = (y_all == c) & (split == sp)

            plt.scatter(
                Z_pca[idx, 0],
                Z_pca[idx, 1],
                s=90 if sp == "support" else 40,
                alpha=0.85 if sp == "support" else 0.55,
                marker=markers[sp],
                label=f"{ID_TO_LABEL[c]}-{sp}"
            )

    plt.title(
        f"{model_name} | Subject {subject} | Epoch {epoch}\n"
        f"Support vs Query PCA"
    )
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()

    if save:
        fname = (
            f"{model_name}_subject_{subject}_epoch_{epoch:02d}_support_query_pca"
            .replace(" ", "_")
            .replace("/", "_")
        )
        path = os.path.join(EMBED_PLOT_DIR, fname + ".png")
        plt.savefig(path, dpi=200)
        print("Saved support/query plot:", path)

    plt.show()


def prototype_distance_report(Z_support, y_support, Z_query, y_query, subject, model_name, epoch):
    Z_support_t = torch.tensor(Z_support, dtype=torch.float32)
    Z_query_t = torch.tensor(Z_query, dtype=torch.float32)
    y_support_t = torch.tensor(y_support, dtype=torch.long)
    y_query_t = torch.tensor(y_query, dtype=torch.long)

    Z_support_t = F.normalize(Z_support_t, dim=1)
    Z_query_t = F.normalize(Z_query_t, dim=1)

    prototypes = []

    for c in range(N_CLASSES):
        proto = Z_support_t[y_support_t == c].mean(dim=0)
        proto = F.normalize(proto, dim=0)
        prototypes.append(proto)

    prototypes = torch.stack(prototypes, dim=0)

    sims = Z_query_t @ prototypes.T
    preds = sims.argmax(dim=1)

    print("\n" + "-" * 80)
    print(f"PROTOTYPE DISTANCE REPORT | {model_name} | Subject {subject} | Epoch {epoch}")
    print("-" * 80)

    for c in range(N_CLASSES):
        idx = y_query_t == c
        if idx.sum() == 0:
            continue

        true_class_sims = sims[idx, c]
        pred_correct = preds[idx] == c

        print(
            f"{ID_TO_LABEL[c]} | "
            f"n={idx.sum().item()} | "
            f"mean_sim_to_own_proto={true_class_sims.mean().item():.4f} | "
            f"correct={pred_correct.sum().item()}/{idx.sum().item()}"
        )

    print("Mean max similarity:", sims.max(dim=1).values.mean().item())
    print("Pred counts:", np.bincount(preds.numpy(), minlength=N_CLASSES))


# ============================================================
# SPLITTING
# ============================================================

def make_fewshot_split(test_indices, y, k_per_class=5, seed=42):
    rng = np.random.default_rng(seed)

    support_idx = []
    query_idx = []

    print("  Few-shot split:")

    for c in range(N_CLASSES):
        cls_idx = test_indices[y[test_indices] == c]
        rng.shuffle(cls_idx)

        k = min(k_per_class, len(cls_idx) // 2)

        support_idx.extend(cls_idx[:k])
        query_idx.extend(cls_idx[k:])

        percent = 100 * k / len(cls_idx) if len(cls_idx) > 0 else 0

        print(
            f"    class {c} ({ID_TO_LABEL[c]}): "
            f"total={len(cls_idx)}, support={k}, query={len(cls_idx)-k}, support%={percent:.1f}%"
        )

    return np.array(support_idx), np.array(query_idx)

# ============================================================
# BASELINE MAJORITY CLASS
# ============================================================

def majority_baseline(y_query):
    vals, counts = np.unique(y_query, return_counts=True)
    majority = vals[np.argmax(counts)]
    preds = np.full_like(y_query, fill_value=majority)

    acc = accuracy_score(y_query, preds)
    bacc = balanced_accuracy_score(y_query, preds)
    cm = confusion_matrix(y_query, preds, labels=[0,1,2])

    return acc, bacc, cm

# ============================================================
# CNN ONLY LOSO
# ============================================================

def run_loso_cnn_only():
    print("\n" + "="*100)
    print("RUNNING BALANCED BASELINE 1: CNN ONLY")
    print("="*100)

    fold_results = []
    unique_subjects = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#"*100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#"*100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx] == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED+fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query")
            continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)

        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        X_train_raw = X[train_idx]
        X_support_raw = X[support_idx]
        X_query_raw = X[query_idx]

        train_mean = X_train_raw.mean(axis=(0,1), keepdims=True)
        train_std = X_train_raw.std(axis=(0,1), keepdims=True) + 1e-6

        X_train = (X_train_raw - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query = (X_query_raw - train_mean) / train_std

        y_train = y[train_idx]
        y_support = y[support_idx]
        y_query = y[query_idx]

        train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader = make_loader(X_query, y_query, batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNN(n_channels=8, n_classes=3, emb_dim=128).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = 0.0
        best_fs_bacc = 0.0
        best_zs_acc = 0.0
        best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS+1):
            model.train()
            total_loss = 0.0
            all_preds, all_true = [], []

            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)

                preds = logits.argmax(dim=1)
                all_preds.extend(preds.detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(all_true, all_preds)
            train_bacc = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_cnn(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_cnn(model, support_loader)
            Z_q, y_q_t = get_embeddings_cnn(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc = max(best_fs_acc, fs_acc)
            best_fs_bacc = max(best_fs_bacc, fs_bacc)
            best_zs_acc = max(best_zs_acc, zs_acc)
            best_zs_bacc = max(best_zs_bacc, zs_bacc)

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.4f} | "
                f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                f"zero_shot_acc={zs_acc:.4f} | zero_shot_bacc={zs_bacc:.4f} | "
                f"fewshot_acc={fs_acc:.4f} | fewshot_bacc={fs_bacc:.4f} | "
                f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}"
            )

            print("  Zero-shot CM:")
            print(zs_cm)
            print("  Few-shot prototype CM:")
            print(fs_cm)

        print("\nFINAL FOLD RESULT")
        print(
            f"Subject {test_subj} | "
            f"best zero-shot={best_zs_acc:.4f} | best zero-shot bacc={best_zs_bacc:.4f} | "
            f"best few-shot={best_fs_acc:.4f} | best few-shot bacc={best_fs_bacc:.4f}"
        )

        fold_results.append({
            "subject": test_subj,
            "best_zero_shot": best_zs_acc,
            "best_zero_shot_bacc": best_zs_bacc,
            "best_fewshot": best_fs_acc,
            "best_fewshot_bacc": best_fs_bacc,
            "majority_acc": maj_acc,
            "majority_bacc": maj_bacc,
            "n_train": len(train_idx),
            "n_support": len(support_idx),
            "n_query": len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx)
        })

    results_df = pd.DataFrame(fold_results)

    print("\n" + "="*100)
    print("BALANCED CNN ONLY LOSO SUMMARY")
    print("="*100)
    display(results_df)

    print("Mean zero-shot:", results_df["best_zero_shot"].mean())
    print("Mean zero-shot bacc:", results_df["best_zero_shot_bacc"].mean())
    print("Mean few-shot:", results_df["best_fewshot"].mean())
    print("Mean few-shot bacc:", results_df["best_fewshot_bacc"].mean())
    print("Mean majority acc:", results_df["majority_acc"].mean())
    print("Mean support %:", results_df["support_percent_total"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_only_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# CNN + FEATURE FUSION LOSO
# ============================================================

def run_loso_cnn_feature_fusion():
    print("\n" + "="*100)
    print("RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION")
    print("="*100)

    fold_results = []
    unique_subjects = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#"*100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#"*100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx] == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED+fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query")
            continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)

        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        X_train_raw = X[train_idx]
        X_support_raw = X[support_idx]
        X_query_raw = X[query_idx]

        F_train_raw = F_all[train_idx]
        F_support_raw = F_all[support_idx]
        F_query_raw = F_all[query_idx]

        train_mean = X_train_raw.mean(axis=(0,1), keepdims=True)
        train_std = X_train_raw.std(axis=(0,1), keepdims=True) + 1e-6

        X_train = (X_train_raw - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query = (X_query_raw - train_mean) / train_std

        feat_mean = F_train_raw.mean(axis=0, keepdims=True)
        feat_std = F_train_raw.std(axis=0, keepdims=True) + 1e-6

        F_train = (F_train_raw - feat_mean) / feat_std
        F_support = (F_support_raw - feat_mean) / feat_std
        F_query = (F_query_raw - feat_mean) / feat_std

        y_train = y[train_idx]
        y_support = y[support_idx]
        y_query = y[query_idx]

        train_loader = make_loader(X_train, y_train, features=F_train, batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, features=F_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader = make_loader(X_query, y_query, features=F_query, batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNNFeatureFusion(
            n_channels=8,
            n_features=F_all.shape[1],
            n_classes=3,
            emb_dim=128,
            feat_dim=64
        ).to(device)

        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = 0.0
        best_fs_bacc = 0.0
        best_zs_acc = 0.0
        best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS+1):
            model.train()
            total_loss = 0.0
            all_preds, all_true = [], []

            for xb, fb, yb in train_loader:
                xb = xb.to(device)
                fb = fb.to(device)
                yb = yb.to(device)

                opt.zero_grad()
                logits, _ = model(xb, fb)
                loss = criterion(logits, yb)
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)

                preds = logits.argmax(dim=1)
                all_preds.extend(preds.detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc = accuracy_score(all_true, all_preds)
            train_bacc = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_fusion(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_fusion(model, support_loader)
            Z_q, y_q_t = get_embeddings_fusion(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)


            # ====================================================
            # EMBEDDING DIAGNOSTICS
            # Plot early, middle, late, and whenever few-shot improves
            # ====================================================
             improved_fs = fs_acc > best_fs_acc

            should_plot = (
                epoch in [1, 3, 5, 10, 15] or
                improved_fs
            )

            if should_plot:
                Z_sup_np = Z_sup.numpy()
                y_sup_np = y_sup_t.numpy()
                Z_q_np = Z_q.numpy()
                y_q_np = y_q_t.numpy()

                Z_all = np.concatenate([Z_sup_np, Z_q_np], axis=0)
                y_all = np.concatenate([y_sup_np, y_q_np], axis=0)

                plot_embeddings_pca_tsne(
                    Z_all,
                    y_all,
                    subject=test_subj,
                    model_name="cnn_feature_fusion",
                    epoch=epoch,
                    split_name="heldout_support_plus_query"
                )

                plot_support_query_embeddings(
                    Z_sup_np,
                    y_sup_np,
                    Z_q_np,
                    y_q_np,
                    subject=test_subj,
                    model_name="cnn_feature_fusion",
                    epoch=epoch
                )

                prototype_distance_report(
                    Z_sup_np,
                    y_sup_np,
                    Z_q_np,
                    y_q_np,
                    subject=test_subj,
                    model_name="cnn_feature_fusion",
                    epoch=epoch
                )


            improved_fs = fs_acc > best_fs_acc
            best_fs_acc = max(best_fs_acc, fs_acc)
            best_fs_bacc = max(best_fs_bacc, fs_bacc)
            best_zs_acc = max(best_zs_acc, zs_acc)
            best_zs_bacc = max(best_zs_bacc, zs_bacc)


            # improved_fs = fs_acc > best_fs_acc


            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.4f} | "
                f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                f"zero_shot_acc={zs_acc:.4f} | zero_shot_bacc={zs_bacc:.4f} | "
                f"fewshot_acc={fs_acc:.4f} | fewshot_bacc={fs_bacc:.4f} | "
                f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}"
            )

            print("  Zero-shot CM:")
            print(zs_cm)
            print("  Few-shot prototype CM:")
            print(fs_cm)

        print("\nFINAL FOLD RESULT")
        print(
            f"Subject {test_subj} | "
            f"best zero-shot={best_zs_acc:.4f} | best zero-shot bacc={best_zs_bacc:.4f} | "
            f"best few-shot={best_fs_acc:.4f} | best few-shot bacc={best_fs_bacc:.4f}"
        )

        fold_results.append({
            "subject": test_subj,
            "best_zero_shot": best_zs_acc,
            "best_zero_shot_bacc": best_zs_bacc,
            "best_fewshot": best_fs_acc,
            "best_fewshot_bacc": best_fs_bacc,
            "majority_acc": maj_acc,
            "majority_bacc": maj_bacc,
            "n_train": len(train_idx),
            "n_support": len(support_idx),
            "n_query": len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx)
        })

    results_df = pd.DataFrame(fold_results)

    print("\n" + "="*100)
    print("BALANCED CNN + FEATURE FUSION LOSO SUMMARY")
    print("="*100)
    display(results_df)

    print("Mean zero-shot:", results_df["best_zero_shot"].mean())
    print("Mean zero-shot bacc:", results_df["best_zero_shot_bacc"].mean())
    print("Mean few-shot:", results_df["best_fewshot"].mean())
    print("Mean few-shot bacc:", results_df["best_fewshot_bacc"].mean())
    print("Mean majority acc:", results_df["majority_acc"].mean())
    print("Mean support %:", results_df["support_percent_total"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_feature_fusion_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# RUN
# ============================================================




IndentationError: unexpected indent (813024998.py, line 1070)

In [ ]:
# ============================================================
# BALANCED LOSO EEG BASELINE  —  WITH EMBEDDING VISUALIZATIONS
# Drops invalid subject 7
# Balances low/mid/high windows WITHIN EACH SUBJECT
# Runs CNN-only first, then CNN + handcrafted feature fusion
#
# NEW: embedding visualizations are produced at every fold:
#   1. Train-split embeddings (PCA + t-SNE)           after final epoch
#   2. Support + query embeddings (PCA + t-SNE)        after final epoch
#   3. Support vs query overlay (PCA)                  after final epoch
#   4. Prototype distance report                       after final epoch
#   5. ALL-SUBJECTS grid (final epoch, query split)    at the end of each run
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, random
import numpy as np
import pandas as pd

from scipy.signal import welch
from scipy.stats import entropy

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import matplotlib
matplotlib.use("Agg")          # safe for Colab / headless
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ============================================================
# CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

SAVE_DIR = "/content/drive/MyDrive/eeg_qc_outputs"
NPZ_PATH = os.path.join(SAVE_DIR, "clean_eeg_windows_qc.npz")

FS = 250
N_CLASSES = 3
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
FEW_SHOT_PER_CLASS = 5

DROP_SUBJECTS = [7]

LABEL_REMAP = {
    "lowlevel": 0,
    "midlevel": 1,
    "highlevel": 2,
}

ID_TO_LABEL = {
    0: "lowlevel",
    1: "midlevel",
    2: "highlevel",
}

CLASS_COLORS = {0: "#4e79a7", 1: "#f28e2b", 2: "#e15759"}
CLASS_MARKERS = {0: "o", 1: "s", 2: "^"}

EMBED_PLOT_DIR = os.path.join(SAVE_DIR, "embedding_diagnostics")
os.makedirs(EMBED_PLOT_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

data = np.load(NPZ_PATH, allow_pickle=True)

X_raw_all     = data["X"].astype(np.float32)
labels_all    = data["labels"]
subjects_all  = data["subjects"]
tasks_all     = data["tasks"]
file_paths_all = data["file_paths"]
start_secs_all = data["start_secs"]

print("Original X:", X_raw_all.shape)
print("Tasks:",    np.unique(tasks_all))
print("Labels:",   np.unique(labels_all))
print("Subjects:", np.unique(subjects_all))

mask = (
    (tasks_all == "Arithmetic") &
    np.isin(labels_all, ["lowlevel", "midlevel", "highlevel"]) &
    (~np.isin(subjects_all, DROP_SUBJECTS))
)

X_raw     = X_raw_all[mask]
labels    = labels_all[mask]
subjects  = subjects_all[mask]
file_paths = file_paths_all[mask]
start_secs = start_secs_all[mask]

y = np.array([LABEL_REMAP[l] for l in labels])

print("\nAfter Arithmetic non-natural + drop subject 7:")
print("X:", X_raw.shape)
print("Subjects:", np.unique(subjects))
print("Label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

# ============================================================
# BALANCE WITHIN EACH SUBJECT
# ============================================================

def balance_within_subjects(X, y, subjects, labels, file_paths, start_secs, seed=42):
    rng = np.random.default_rng(seed)
    keep_indices = []

    print("\n" + "=" * 100)
    print("BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT")
    print("=" * 100)

    for s in sorted(np.unique(subjects)):
        subj_idx = np.where(subjects == s)[0]
        class_counts = {c: int(np.sum(y[subj_idx] == c)) for c in range(N_CLASSES)}
        min_count = min(class_counts.values())
        print(f"Subject {s}: raw counts={class_counts} -> keeping {min_count}/class")

        if min_count == 0:
            print(f"  SKIPPING subject {s}: missing a class")
            continue

        for c in range(N_CLASSES):
            cls_idx = subj_idx[y[subj_idx] == c]
            chosen  = rng.choice(cls_idx, size=min_count, replace=False)
            keep_indices.extend(chosen.tolist())

    keep_indices = np.array(sorted(keep_indices))
    return (
        X[keep_indices], y[keep_indices], subjects[keep_indices],
        labels[keep_indices], file_paths[keep_indices],
        start_secs[keep_indices], keep_indices
    )

X, y, subjects, labels, file_paths, start_secs, kept_original_indices = balance_within_subjects(
    X_raw, y, subjects, labels, file_paths, start_secs, seed=SEED
)

print("\nBalanced X:", X.shape)
print("Balanced global label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

balanced_summary = []
for s in sorted(np.unique(subjects)):
    row = {"subject": s}
    for c in range(N_CLASSES):
        row[ID_TO_LABEL[c]] = int(np.sum((subjects == s) & (y == c)))
    row["total"] = int(np.sum(subjects == s))
    balanced_summary.append(row)

balanced_summary = pd.DataFrame(balanced_summary)
display(balanced_summary)

# ============================================================
# HANDCRAFTED FEATURES
# ============================================================

bands = {
    "theta": (4,  8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 35),
}

def bandpower_from_psd(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return np.trapezoid(pxx[idx], f[idx])

def spectral_entropy_from_psd(pxx):
    p = pxx / (np.sum(pxx) + 1e-12)
    return entropy(p)

def extract_features_one_window(w, fs=250):
    feats = []

    ch_mean = np.mean(w, axis=0);  ch_std = np.std(w, axis=0)
    ch_var  = np.var(w,  axis=0);  ch_ptp = np.ptp(w, axis=0)

    feats.extend(ch_mean.tolist()); feats.extend(ch_std.tolist())
    feats.extend(ch_var.tolist());  feats.extend(ch_ptp.tolist())

    feats.append(np.mean(ch_var)); feats.append(np.max(ch_var))
    feats.append(np.mean(ch_ptp)); feats.append(np.max(ch_ptp))

    band_means = {b: [] for b in bands}
    spec_ents  = []

    for ch in range(w.shape[1]):
        f, pxx = welch(w[:, ch], fs=fs, nperseg=512)
        for name, (lo, hi) in bands.items():
            band_means[name].append(bandpower_from_psd(f, pxx, lo, hi))
        spec_ents.append(spectral_entropy_from_psd(pxx))

    for name in ["theta", "alpha", "beta", "gamma"]:
        vals = np.array(band_means[name])
        feats.append(np.mean(vals))
        feats.append(np.log(np.mean(vals) + 1e-12))
        feats.append(np.std(vals))

    theta = np.mean(band_means["theta"]); alpha = np.mean(band_means["alpha"])
    beta  = np.mean(band_means["beta"]);  gamma = np.mean(band_means["gamma"])

    feats.append(theta / (alpha + 1e-12)); feats.append(theta / (beta + 1e-12))
    feats.append(alpha / (beta  + 1e-12)); feats.append(gamma / (beta + 1e-12))

    feats.append(np.log(theta + 1e-12) - np.log(alpha + 1e-12))
    feats.append(np.log(theta + 1e-12) - np.log(beta  + 1e-12))
    feats.append(np.log(alpha + 1e-12) - np.log(beta  + 1e-12))

    feats.append(np.mean(spec_ents)); feats.append(np.std(spec_ents))

    return np.array(feats, dtype=np.float32)

FEATURE_CACHE = os.path.join(SAVE_DIR, "arithmetic_balanced_baseline_features.npy")

if os.path.exists(FEATURE_CACHE):
    print("\nLoading cached balanced handcrafted features...")
    F_all = np.load(FEATURE_CACHE)
    if len(F_all) != len(X):
        print("Cached feature size mismatch. Recomputing.")
        os.remove(FEATURE_CACHE)
        F_all = None
else:
    F_all = None

if F_all is None:
    print("\nExtracting handcrafted features for BALANCED dataset...")
    feats = []
    for i in range(len(X)):
        if i % 100 == 0:
            print(f"  extracting {i}/{len(X)}")
        feats.append(extract_features_one_window(X[i], FS))
    F_all = np.stack(feats).astype(np.float32)
    np.save(FEATURE_CACHE, F_all)
    print("Saved features:", FEATURE_CACHE)

print("Feature matrix:", F_all.shape)

# ============================================================
# DATASET
# ============================================================

class EEGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, features=None):
        self.X = X; self.y = y.astype(np.int64); self.features = features

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x     = torch.tensor(self.X[idx].T, dtype=torch.float32)
        label = torch.tensor(self.y[idx],   dtype=torch.long)
        if self.features is None:
            return x, label
        return x, torch.tensor(self.features[idx], dtype=torch.float32), label


def make_loader(X, y, features=None, batch_size=64, shuffle=False, weighted=False):
    ds = EEGDataset(X, y, features)
    if weighted:
        class_counts  = np.bincount(y, minlength=N_CLASSES)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[y]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights), replacement=True
        )
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=False)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

# ============================================================
# MODELS
# ============================================================

class EEGCNN(nn.Module):
    def __init__(self, n_channels=8, n_classes=3, emb_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32,  kernel_size=15, padding=7), nn.BatchNorm1d(32),  nn.GELU(),
            nn.Conv1d(32,         64,  kernel_size=15, padding=7), nn.BatchNorm1d(64),  nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64,         128, kernel_size=9,  padding=4), nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(128,        128, kernel_size=7,  padding=3), nn.BatchNorm1d(128), nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(nn.Linear(128, emb_dim), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x):
        h = self.encoder(x); h = self.pool(h).squeeze(-1); return self.proj(h)

    def forward(self, x):
        z = self.forward_features(x); return self.classifier(z), z


class EEGCNNFeatureFusion(nn.Module):
    def __init__(self, n_channels=8, n_features=1, n_classes=3, emb_dim=128, feat_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32,  kernel_size=15, padding=7), nn.BatchNorm1d(32),  nn.GELU(),
            nn.Conv1d(32,         64,  kernel_size=15, padding=7), nn.BatchNorm1d(64),  nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64,         128, kernel_size=9,  padding=4), nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(128,        128, kernel_size=7,  padding=3), nn.BatchNorm1d(128), nn.GELU(),
        )
        self.pool     = nn.AdaptiveAvgPool1d(1)
        self.raw_proj = nn.Sequential(nn.Linear(128, emb_dim), nn.GELU(), nn.Dropout(0.2))
        self.feat_proj = nn.Sequential(
            nn.Linear(n_features, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, feat_dim), nn.GELU()
        )
        self.fusion     = nn.Sequential(nn.Linear(emb_dim + feat_dim, emb_dim), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x, features):
        h      = self.encoder(x); h = self.pool(h).squeeze(-1)
        z_raw  = self.raw_proj(h)
        z_feat = self.feat_proj(features)
        return self.fusion(torch.cat([z_raw, z_feat], dim=1))

    def forward(self, x, features):
        z = self.forward_features(x, features); return self.classifier(z), z

# ============================================================
# EVAL HELPERS
# ============================================================

@torch.no_grad()
def eval_classifier_cnn(model, loader):
    model.eval(); all_preds, all_true = [], []
    for x, yb in loader:
        logits, _ = model(x.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())
    acc  = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm   = confusion_matrix(all_true, all_preds, labels=[0, 1, 2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def eval_classifier_fusion(model, loader):
    model.eval(); all_preds, all_true = [], []
    for x, f, yb in loader:
        logits, _ = model(x.to(device), f.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())
    acc  = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm   = confusion_matrix(all_true, all_preds, labels=[0, 1, 2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def get_embeddings_cnn(model, loader):
    model.eval(); Z, Y = [], []
    for x, yb in loader:
        _, z = model(x.to(device)); Z.append(z.cpu()); Y.append(yb)
    return torch.cat(Z), torch.cat(Y)

@torch.no_grad()
def get_embeddings_fusion(model, loader):
    model.eval(); Z, Y = [], []
    for x, f, yb in loader:
        _, z = model(x.to(device), f.to(device)); Z.append(z.cpu()); Y.append(yb)
    return torch.cat(Z), torch.cat(Y)

def prototype_eval(Z_support, y_support, Z_query, y_query):
    Z_support = F.normalize(Z_support, dim=1)
    Z_query   = F.normalize(Z_query,   dim=1)
    prototypes = []
    for c in range(N_CLASSES):
        mask  = y_support == c
        proto = Z_support[mask].mean(0) if mask.sum() > 0 else torch.zeros(Z_support.shape[1])
        prototypes.append(F.normalize(proto, dim=0))
    prototypes = torch.stack(prototypes)
    preds      = (Z_query @ prototypes.T).argmax(1)
    acc  = accuracy_score(y_query.numpy(), preds.numpy())
    bacc = balanced_accuracy_score(y_query.numpy(), preds.numpy())
    cm   = confusion_matrix(y_query.numpy(), preds.numpy(), labels=[0, 1, 2])
    return acc, bacc, cm, preds.numpy()

# ============================================================
# SPLITTING / BASELINE
# ============================================================

def make_fewshot_split(test_indices, y, k_per_class=5, seed=42):
    rng = np.random.default_rng(seed)
    support_idx, query_idx = [], []
    print("  Few-shot split:")
    for c in range(N_CLASSES):
        cls_idx = test_indices[y[test_indices] == c]
        rng.shuffle(cls_idx)
        k = min(k_per_class, len(cls_idx) // 2)
        support_idx.extend(cls_idx[:k])
        query_idx.extend(cls_idx[k:])
        pct = 100 * k / len(cls_idx) if len(cls_idx) > 0 else 0
        print(f"    class {c} ({ID_TO_LABEL[c]}): total={len(cls_idx)}, "
              f"support={k}, query={len(cls_idx)-k}, support%={pct:.1f}%")
    return np.array(support_idx), np.array(query_idx)


def majority_baseline(y_query):
    vals, counts = np.unique(y_query, return_counts=True)
    majority = vals[np.argmax(counts)]
    preds    = np.full_like(y_query, fill_value=majority)
    return (accuracy_score(y_query, preds),
            balanced_accuracy_score(y_query, preds),
            confusion_matrix(y_query, preds, labels=[0, 1, 2]))

# ============================================================
# ============================================================
#   EMBEDDING VISUALIZATION UTILITIES
# ============================================================
# ============================================================

def _fit_pca_tsne(Z_np, seed=SEED):
    """Return (Z_pca, Z_tsne, pca_var) after standard scaling."""
    Z_sc = StandardScaler().fit_transform(Z_np)

    pca   = PCA(n_components=2, random_state=seed)
    Z_pca = pca.fit_transform(Z_sc)

    n_pts  = len(Z_sc)
    perp   = max(3, min(30, (n_pts - 1) // 3))
    tsne   = TSNE(n_components=2, perplexity=perp, init="pca",
                  learning_rate="auto", random_state=seed)
    Z_tsne = tsne.fit_transform(Z_sc)

    return Z_pca, Z_tsne, pca.explained_variance_ratio_.sum(), perp


def _scatter_2d(ax, Z_2d, Y, title, xlabel, ylabel, show_legend=True):
    for c in range(N_CLASSES):
        idx = Y == c
        ax.scatter(Z_2d[idx, 0], Z_2d[idx, 1],
                   s=40, alpha=0.75, label=ID_TO_LABEL[c],
                   color=CLASS_COLORS[c], marker=CLASS_MARKERS[c])
    ax.set_title(title, fontsize=9, pad=4)
    ax.set_xlabel(xlabel, fontsize=8); ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(fontsize=7, markerscale=1.2)


def save_fig(fig, fname):
    path = os.path.join(EMBED_PLOT_DIR, fname + ".png")
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved:", path)
    plt.close(fig)


# ------------------------------------------------------------------
# 1.  Per-fold: train split + query split (PCA + t-SNE side by side)
# ------------------------------------------------------------------
def plot_fold_embeddings(Z_train_np, y_train_np,
                         Z_query_np, y_query_np,
                         subject, model_name, epoch):
    """
    2-row × 2-col figure:
      row 0 = train split   (PCA | t-SNE)
      row 1 = query split   (PCA | t-SNE)
    """
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    fig.suptitle(f"{model_name}  |  Subject {subject} held-out  |  Epoch {epoch}",
                 fontsize=11, fontweight="bold")

    for row, (Z_np, Y_np, split_name) in enumerate([
        (Z_train_np, y_train_np, "Train"),
        (Z_query_np, y_query_np, "Query (held-out)"),
    ]):
        Z_pca, Z_tsne, var, perp = _fit_pca_tsne(Z_np)

        _scatter_2d(axes[row, 0], Z_pca, Y_np,
                    title=f"{split_name} — PCA  (var={var:.3f})",
                    xlabel="PC1", ylabel="PC2")
        _scatter_2d(axes[row, 1], Z_tsne, Y_np,
                    title=f"{split_name} — t-SNE  (perp={perp})",
                    xlabel="t-SNE 1", ylabel="t-SNE 2")

    plt.tight_layout()
    save_fig(fig, f"{model_name}_subj{subject}_ep{epoch:02d}_train_query_embed")


# ------------------------------------------------------------------
# 2.  Per-fold: support vs query overlay (PCA only)
# ------------------------------------------------------------------
def plot_support_query_overlay(Z_support_np, y_support_np,
                               Z_query_np,   y_query_np,
                               subject, model_name, epoch):
    Z_all = np.concatenate([Z_support_np, Z_query_np], axis=0)
    y_all = np.concatenate([y_support_np, y_query_np], axis=0)
    split = np.array(["support"] * len(Z_support_np) + ["query"] * len(Z_query_np))

    Z_sc  = StandardScaler().fit_transform(Z_all)
    pca   = PCA(n_components=2, random_state=SEED)
    Z_pca = pca.fit_transform(Z_sc)

    fig, ax = plt.subplots(figsize=(6, 5))
    for c in range(N_CLASSES):
        for sp, marker, sz, alpha in [("support", "X", 120, 0.95),
                                       ("query",   "o",  40, 0.55)]:
            idx = (y_all == c) & (split == sp)
            ax.scatter(Z_pca[idx, 0], Z_pca[idx, 1],
                       s=sz, alpha=alpha, marker=marker,
                       color=CLASS_COLORS[c],
                       label=f"{ID_TO_LABEL[c]} ({sp})" if c == 0 or sp == "query" else "")

    # Custom legend
    from matplotlib.lines import Line2D
    handles = []
    for c in range(N_CLASSES):
        handles.append(Line2D([0], [0], marker="X", color="w",
                               markerfacecolor=CLASS_COLORS[c], markersize=9,
                               label=f"{ID_TO_LABEL[c]} support"))
        handles.append(Line2D([0], [0], marker="o", color="w",
                               markerfacecolor=CLASS_COLORS[c], markersize=7, alpha=0.7,
                               label=f"{ID_TO_LABEL[c]} query"))
    ax.legend(handles=handles, fontsize=7, ncol=2)

    ax.set_title(f"{model_name}  |  Subject {subject}  |  Epoch {epoch}\n"
                 f"Support (✕) vs Query (●) — PCA  (var={pca.explained_variance_ratio_.sum():.3f})",
                 fontsize=9)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    plt.tight_layout()
    save_fig(fig, f"{model_name}_subj{subject}_ep{epoch:02d}_support_query_overlay")


# ------------------------------------------------------------------
# 3.  All-subjects grid  (query embeddings, PCA + t-SNE)
# ------------------------------------------------------------------
def plot_all_subjects_grid(all_subject_embeddings, model_name):
    """
    all_subject_embeddings : list of dicts
        {"subject": int, "Z": np.ndarray, "Y": np.ndarray,
         "fs_acc": float, "zs_acc": float}

    Produces two figures:
      - one 2-col grid (PCA | t-SNE) with one row per subject
      - saved to disk
    """
    n_subj = len(all_subject_embeddings)
    if n_subj == 0:
        return

    fig, axes = plt.subplots(n_subj, 2,
                              figsize=(11, 4.5 * n_subj),
                              squeeze=False)
    fig.suptitle(f"{model_name}  —  All Subjects  (query split, final epoch)",
                 fontsize=13, fontweight="bold", y=1.002)

    for row, entry in enumerate(all_subject_embeddings):
        subj     = entry["subject"]
        Z_np     = entry["Z"]
        Y_np     = entry["Y"]
        fs_acc   = entry["fs_acc"]
        zs_acc   = entry["zs_acc"]

        Z_pca, Z_tsne, var, perp = _fit_pca_tsne(Z_np)

        _scatter_2d(axes[row, 0], Z_pca, Y_np,
                    title=(f"Subj {subj} — PCA (var={var:.3f}) | "
                           f"ZS={zs_acc:.3f} FS={fs_acc:.3f}"),
                    xlabel="PC1", ylabel="PC2",
                    show_legend=(row == 0))

        _scatter_2d(axes[row, 1], Z_tsne, Y_np,
                    title=(f"Subj {subj} — t-SNE (perp={perp}) | "
                           f"ZS={zs_acc:.3f} FS={fs_acc:.3f}"),
                    xlabel="t-SNE 1", ylabel="t-SNE 2",
                    show_legend=False)

    # shared legend on top
    handles = [plt.Line2D([0], [0], marker=CLASS_MARKERS[c], color="w",
                           markerfacecolor=CLASS_COLORS[c], markersize=9,
                           label=ID_TO_LABEL[c]) for c in range(N_CLASSES)]
    fig.legend(handles=handles, loc="upper right", fontsize=9,
               bbox_to_anchor=(1.0, 1.0), frameon=True)

    plt.tight_layout()
    save_fig(fig, f"{model_name}_ALL_SUBJECTS_query_embed_grid")


# ------------------------------------------------------------------
# 4.  Prototype distance report (printed)
# ------------------------------------------------------------------
def prototype_distance_report(Z_support_t, y_support_t,
                               Z_query_t,   y_query_t,
                               subject, model_name, epoch):
    Zs = F.normalize(Z_support_t.float(), dim=1)
    Zq = F.normalize(Z_query_t.float(),   dim=1)
    protos = []
    for c in range(N_CLASSES):
        m     = y_support_t == c
        proto = Zs[m].mean(0) if m.sum() > 0 else torch.zeros(Zs.shape[1])
        protos.append(F.normalize(proto, dim=0))
    protos = torch.stack(protos)
    sims   = Zq @ protos.T
    preds  = sims.argmax(1)

    print(f"\n{'─'*80}")
    print(f"PROTOTYPE DISTANCES | {model_name} | Subject {subject} | Epoch {epoch}")
    print(f"{'─'*80}")
    for c in range(N_CLASSES):
        idx  = y_query_t == c
        if idx.sum() == 0: continue
        own  = sims[idx, c]
        corr = (preds[idx] == c)
        print(f"  {ID_TO_LABEL[c]:<10}  n={idx.sum().item():3d}  "
              f"sim_own={own.mean().item():.4f}  "
              f"correct={corr.sum().item()}/{idx.sum().item()}")
    print(f"  Mean max-sim: {sims.max(1).values.mean().item():.4f}")
    print(f"  Pred counts: {np.bincount(preds.numpy(), minlength=N_CLASSES)}")


# ============================================================
# CNN ONLY LOSO
# ============================================================

def run_loso_cnn_only():
    print("\n" + "=" * 100)
    print("RUNNING BALANCED BASELINE 1: CNN ONLY")
    print("=" * 100)

    fold_results            = []
    all_subject_embeddings  = []          # <-- collect for grid
    unique_subjects         = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#" * 100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#" * 100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx  = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx]  == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED + fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query"); continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)
        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        # --- normalise ---
        X_train_raw   = X[train_idx];   X_support_raw = X[support_idx]; X_query_raw = X[query_idx]
        train_mean    = X_train_raw.mean(axis=(0, 1), keepdims=True)
        train_std     = X_train_raw.std(axis=(0,  1), keepdims=True) + 1e-6

        X_train   = (X_train_raw   - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query   = (X_query_raw   - train_mean) / train_std

        y_train   = y[train_idx];  y_support = y[support_idx]; y_query = y[query_idx]

        train_loader   = make_loader(X_train,   y_train,   batch_size=BATCH_SIZE, weighted=True)
        support_loader = make_loader(X_support, y_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader   = make_loader(X_query,   y_query,   batch_size=BATCH_SIZE, shuffle=False)
        # we also need a train loader without shuffling for embedding extraction
        train_embed_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=False)

        model     = EEGCNN(n_channels=8, n_classes=3, emb_dim=128).to(device)
        opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = best_fs_bacc = best_zs_acc = best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0; all_preds, all_true = [], []

            for xb, yb in train_loader:
                xb = xb.to(device); yb = yb.to(device)
                opt.zero_grad()
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward(); opt.step()
                total_loss += loss.item() * xb.size(0)
                all_preds.extend(logits.argmax(1).detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss  = total_loss / len(train_loader.dataset)
            train_acc   = accuracy_score(all_true, all_preds)
            train_bacc  = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_cnn(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_cnn(model, support_loader)
            Z_q,   y_q_t  = get_embeddings_cnn(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc  = max(best_fs_acc,  fs_acc);  best_fs_bacc  = max(best_fs_bacc,  fs_bacc)
            best_zs_acc  = max(best_zs_acc,  zs_acc);  best_zs_bacc  = max(best_zs_bacc,  zs_bacc)

            print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | "
                  f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                  f"zero_shot={zs_acc:.4f} | zs_bacc={zs_bacc:.4f} | "
                  f"fewshot={fs_acc:.4f} | fs_bacc={fs_bacc:.4f} | "
                  f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}")
            print("  Zero-shot CM:"); print(zs_cm)
            print("  Few-shot prototype CM:"); print(fs_cm)

        # ---- EMBEDDING VISUALIZATIONS (final epoch) ----
        print(f"\n[VIZ] Generating embedding plots for subject {test_subj}...")

        Z_train_np = get_embeddings_cnn(model, train_embed_loader)[0].numpy()
        y_train_np = y_train
        Z_q_np     = Z_q.numpy()
        y_q_np     = y_q_t.numpy()
        Z_sup_np   = Z_sup.numpy()
        y_sup_np   = y_sup_t.numpy()

        # 1. Train + query PCA/t-SNE
        plot_fold_embeddings(Z_train_np, y_train_np,
                             Z_q_np,    y_q_np,
                             subject=test_subj, model_name="CNN_only", epoch=EPOCHS)

        # 2. Support vs query overlay
        plot_support_query_overlay(Z_sup_np, y_sup_np,
                                   Z_q_np,   y_q_np,
                                   subject=test_subj, model_name="CNN_only", epoch=EPOCHS)

        # 3. Prototype distance report
        prototype_distance_report(Z_sup, y_sup_t, Z_q, y_q_t,
                                  subject=test_subj, model_name="CNN_only", epoch=EPOCHS)

        # 4. Collect for all-subjects grid
        all_subject_embeddings.append({
            "subject": test_subj,
            "Z": Z_q_np,
            "Y": y_q_np,
            "fs_acc": best_fs_acc,
            "zs_acc": best_zs_acc,
        })

        print("\nFINAL FOLD RESULT")
        print(f"Subject {test_subj} | best ZS={best_zs_acc:.4f} | ZS_bacc={best_zs_bacc:.4f} | "
              f"FS={best_fs_acc:.4f} | FS_bacc={best_fs_bacc:.4f}")

        fold_results.append({
            "subject":              test_subj,
            "best_zero_shot":       best_zs_acc,
            "best_zero_shot_bacc":  best_zs_bacc,
            "best_fewshot":         best_fs_acc,
            "best_fewshot_bacc":    best_fs_bacc,
            "majority_acc":         maj_acc,
            "majority_bacc":        maj_bacc,
            "n_train":              len(train_idx),
            "n_support":            len(support_idx),
            "n_query":              len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx),
        })

    # ---- ALL-SUBJECTS GRID ----
    print("\n[VIZ] Generating all-subjects embedding grid...")
    plot_all_subjects_grid(all_subject_embeddings, model_name="CNN_only")

    results_df = pd.DataFrame(fold_results)

    print("\n" + "=" * 100)
    print("BALANCED CNN ONLY LOSO SUMMARY")
    print("=" * 100)
    display(results_df)
    print("Mean ZS:        ", results_df["best_zero_shot"].mean())
    print("Mean ZS bacc:   ", results_df["best_zero_shot_bacc"].mean())
    print("Mean FS:        ", results_df["best_fewshot"].mean())
    print("Mean FS bacc:   ", results_df["best_fewshot_bacc"].mean())
    print("Mean majority:  ", results_df["majority_acc"].mean())
    print("Mean support %: ", results_df["support_percent_total"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_only_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# CNN + FEATURE FUSION LOSO
# ============================================================

def run_loso_cnn_feature_fusion():
    print("\n" + "=" * 100)
    print("RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION")
    print("=" * 100)

    fold_results           = []
    all_subject_embeddings = []
    unique_subjects        = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#" * 100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#" * 100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx  = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx]  == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED + fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query"); continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)
        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        # --- raw signals ---
        X_train_raw   = X[train_idx];   X_support_raw = X[support_idx]; X_query_raw = X[query_idx]
        train_mean    = X_train_raw.mean(axis=(0, 1), keepdims=True)
        train_std     = X_train_raw.std(axis=(0,  1), keepdims=True) + 1e-6
        X_train   = (X_train_raw   - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query   = (X_query_raw   - train_mean) / train_std

        # --- handcrafted features ---
        F_train_raw   = F_all[train_idx]; F_support_raw = F_all[support_idx]; F_query_raw = F_all[query_idx]
        feat_mean     = F_train_raw.mean(axis=0, keepdims=True)
        feat_std      = F_train_raw.std(axis=0,  keepdims=True) + 1e-6
        F_train   = (F_train_raw   - feat_mean) / feat_std
        F_support = (F_support_raw - feat_mean) / feat_std
        F_query   = (F_query_raw   - feat_mean) / feat_std

        y_train = y[train_idx]; y_support = y[support_idx]; y_query = y[query_idx]

        train_loader        = make_loader(X_train,   y_train,   features=F_train,   batch_size=BATCH_SIZE, weighted=True)
        support_loader      = make_loader(X_support, y_support, features=F_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader        = make_loader(X_query,   y_query,   features=F_query,   batch_size=BATCH_SIZE, shuffle=False)
        train_embed_loader  = make_loader(X_train,   y_train,   features=F_train,   batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNNFeatureFusion(
            n_channels=8, n_features=F_all.shape[1],
            n_classes=3,  emb_dim=128, feat_dim=64
        ).to(device)

        opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = best_fs_bacc = best_zs_acc = best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0; all_preds, all_true = [], []

            for xb, fb, yb in train_loader:
                xb = xb.to(device); fb = fb.to(device); yb = yb.to(device)
                opt.zero_grad()
                logits, _ = model(xb, fb)
                loss = criterion(logits, yb)
                loss.backward(); opt.step()
                total_loss += loss.item() * xb.size(0)
                all_preds.extend(logits.argmax(1).detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc  = accuracy_score(all_true, all_preds)
            train_bacc = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_fusion(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_fusion(model, support_loader)
            Z_q,   y_q_t  = get_embeddings_fusion(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc  = max(best_fs_acc,  fs_acc);  best_fs_bacc  = max(best_fs_bacc,  fs_bacc)
            best_zs_acc  = max(best_zs_acc,  zs_acc);  best_zs_bacc  = max(best_zs_bacc,  zs_bacc)

            print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | "
                  f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                  f"zero_shot={zs_acc:.4f} | zs_bacc={zs_bacc:.4f} | "
                  f"fewshot={fs_acc:.4f} | fs_bacc={fs_bacc:.4f} | "
                  f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}")
            print("  Zero-shot CM:"); print(zs_cm)
            print("  Few-shot prototype CM:"); print(fs_cm)

        # ---- EMBEDDING VISUALIZATIONS (final epoch) ----
        print(f"\n[VIZ] Generating embedding plots for subject {test_subj}...")

        Z_train_np = get_embeddings_fusion(model, train_embed_loader)[0].numpy()
        y_train_np = y_train
        Z_q_np     = Z_q.numpy()
        y_q_np     = y_q_t.numpy()
        Z_sup_np   = Z_sup.numpy()
        y_sup_np   = y_sup_t.numpy()

        plot_fold_embeddings(Z_train_np, y_train_np,
                             Z_q_np,    y_q_np,
                             subject=test_subj, model_name="CNN_fusion", epoch=EPOCHS)

        plot_support_query_overlay(Z_sup_np, y_sup_np,
                                   Z_q_np,   y_q_np,
                                   subject=test_subj, model_name="CNN_fusion", epoch=EPOCHS)

        prototype_distance_report(Z_sup, y_sup_t, Z_q, y_q_t,
                                  subject=test_subj, model_name="CNN_fusion", epoch=EPOCHS)

        all_subject_embeddings.append({
            "subject": test_subj,
            "Z": Z_q_np,
            "Y": y_q_np,
            "fs_acc": best_fs_acc,
            "zs_acc": best_zs_acc,
        })

        print("\nFINAL FOLD RESULT")
        print(f"Subject {test_subj} | best ZS={best_zs_acc:.4f} | ZS_bacc={best_zs_bacc:.4f} | "
              f"FS={best_fs_acc:.4f} | FS_bacc={best_fs_bacc:.4f}")

        fold_results.append({
            "subject":               test_subj,
            "best_zero_shot":        best_zs_acc,
            "best_zero_shot_bacc":   best_zs_bacc,
            "best_fewshot":          best_fs_acc,
            "best_fewshot_bacc":     best_fs_bacc,
            "majority_acc":          maj_acc,
            "majority_bacc":         maj_bacc,
            "n_train":               len(train_idx),
            "n_support":             len(support_idx),
            "n_query":               len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx),
        })

    # ---- ALL-SUBJECTS GRID ----
    print("\n[VIZ] Generating all-subjects embedding grid...")
    plot_all_subjects_grid(all_subject_embeddings, model_name="CNN_fusion")

    results_df = pd.DataFrame(fold_results)

    print("\n" + "=" * 100)
    print("BALANCED CNN + FEATURE FUSION LOSO SUMMARY")
    print("=" * 100)
    display(results_df)
    print("Mean ZS:        ", results_df["best_zero_shot"].mean())
    print("Mean ZS bacc:   ", results_df["best_zero_shot_bacc"].mean())
    print("Mean FS:        ", results_df["best_fewshot"].mean())
    print("Mean FS bacc:   ", results_df["best_fewshot_bacc"].mean())
    print("Mean majority:  ", results_df["majority_acc"].mean())
    print("Mean support %: ", results_df["support_percent_total"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_feature_fusion_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# RUN
# ============================================================

# cnn_only_results    = run_loso_cnn_only()
fusion_results      = run_loso_cnn_feature_fusion()

Mounted at /content/drive
DEVICE: cuda
Original X: (2371, 1000, 8)
Tasks: ['Arithmetic' 'Stroop']
Labels: ['highlevel' 'lowlevel' 'midlevel' 'natural']
Subjects: [ 1  2  3  5  6  7  8  9 10 11 12 13 14 15]

After Arithmetic non-natural + drop subject 7:
X: (877, 1000, 8)
Subjects: [ 1  2  3  5  6  8  9 10 11 12 13 14 15]
Label counts: {'lowlevel': 240, 'midlevel': 261, 'highlevel': 376}

BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT
Subject 1: raw counts={0: 21, 1: 22, 2: 24} -> keeping 21/class
Subject 2: raw counts={0: 18, 1: 19, 2: 34} -> keeping 18/class
Subject 3: raw counts={0: 13, 1: 19, 2: 22} -> keeping 13/class
Subject 5: raw counts={0: 13, 1: 20, 2: 20} -> keeping 13/class
Subject 6: raw counts={0: 16, 1: 18, 2: 25} -> keeping 16/class
Subject 8: raw counts={0: 35, 1: 34, 2: 40} -> keeping 34/class
Subject 9: raw counts={0: 17, 1: 17, 2: 55} -> keeping 17/class
Subject 10: raw counts={0: 14, 1: 16, 2: 20} -> keeping 14/class
Subject 11: raw counts={0: 21, 1: 22, 2: 28} -> ke

,subject,lowlevel,midlevel,highlevel,total
0,1,21,21,21,63
1,2,18,18,18,54
2,3,13,13,13,39
3,5,13,13,13,39
4,6,16,16,16,48
5,8,34,34,34,102
6,9,17,17,17,51
7,10,14,14,14,42
8,11,21,21,21,63
9,12,16,16,16,48



Loading cached balanced handcrafted features...
Feature matrix: (702, 57)

RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION

####################################################################################################
FOLD 1/13 | HELD OUT SUBJECT = 1
####################################################################################################
Train windows: 639
Held-out windows: 63
Train label counts: {'lowlevel': 213, 'midlevel': 213, 'highlevel': 213}
Held label counts : {'lowlevel': 21, 'midlevel': 21, 'highlevel': 21}
  Few-shot split:
    class 0 (lowlevel): total=21, support=5, query=16, support%=23.8%
    class 1 (midlevel): total=21, support=5, query=16, support%=23.8%
    class 2 (highlevel): total=21, support=5, query=16, support%=23.8%
Majority baseline query acc=0.3333 | balanced_acc=0.3333
[[16  0  0]
 [16  0  0]
 [16  0  0]]
Epoch 01/15 | loss=1.0972 | train_acc=0.3443 | train_bacc=0.3474 | zero_shot=0.3333 | zs_bacc=0.3333 | fewshot=0.4792 | 

,subject,best_zero_shot,best_zero_shot_bacc,best_fewshot,best_fewshot_bacc,majority_acc,majority_bacc,n_train,n_support,n_query,support_percent_total
0,1,0.375000,0.375000,0.520833,0.520833,0.333333,0.333333,639,15,48,23.809524
1,2,0.461538,0.461538,0.461538,0.461538,0.333333,0.333333,648,15,39,27.777778
2,3,0.541667,0.541667,0.791667,0.791667,0.333333,0.333333,663,15,24,38.461538
3,5,0.625000,0.625000,0.541667,0.541667,0.333333,0.333333,663,15,24,38.461538
4,6,0.515152,0.515152,0.363636,0.363636,0.333333,0.333333,654,15,33,31.250000
5,8,0.367816,0.367816,0.448276,0.448276,0.333333,0.333333,600,15,87,14.705882
6,9,0.444444,0.444444,0.638889,0.638889,0.333333,0.333333,651,15,36,29.411765
7,10,0.481481,0.481481,0.481481,0.481481,0.333333,0.333333,660,15,27,35.714286
8,11,0.458333,0.458333,0.500000,0.500000,0.333333,0.333333,639,15,48,23.809524
9,12,0.515152,0.515152,0.515152,0.515152,0.333333,0.333333,654,15,33,31.250000


Mean ZS:         0.46868790900621143
Mean ZS bacc:    0.46868790900621143
Mean FS:         0.527234486949341
Mean FS bacc:    0.527234486949341
Mean majority:   0.3333333333333333
Mean support %:  29.497509582053517
Saved: /content/drive/MyDrive/eeg_qc_outputs/balanced_baseline_cnn_feature_fusion_loso_results.csv


In [ ]:
# ============================================================
# BALANCED LOSO EEG BASELINE  —  WITH EMBEDDING VISUALIZATIONS
# Drops invalid subject 7
# Balances low/mid/high windows WITHIN EACH SUBJECT
# Runs CNN-only first, then CNN + handcrafted feature fusion
#
# NEW: embedding visualizations are produced at every fold:
#   1. Train-split embeddings (PCA + t-SNE)           after final epoch
#   2. Support + query embeddings (PCA + t-SNE)        after final epoch
#   3. Support vs query overlay (PCA)                  after final epoch
#   4. Prototype distance report                       after final epoch
#   5. ALL-SUBJECTS grid (final epoch, query split)    at the end of each run
#
# NEW: intra/inter-class distance tracking per fold & summary plots
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, random
import numpy as np
import pandas as pd

from scipy.signal import welch
from scipy.stats import entropy

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

import matplotlib
matplotlib.use("Agg")          # safe for Colab / headless
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ============================================================
# CONFIG
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

SAVE_DIR = "/content/drive/MyDrive/eeg_qc_outputs"
NPZ_PATH = os.path.join(SAVE_DIR, "clean_eeg_windows_qc.npz")

FS = 250
N_CLASSES = 3
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
FEW_SHOT_PER_CLASS = 5

DROP_SUBJECTS = [7]

LABEL_REMAP = {
    "lowlevel": 0,
    "midlevel": 1,
    "highlevel": 2,
}

ID_TO_LABEL = {
    0: "lowlevel",
    1: "midlevel",
    2: "highlevel",
}

CLASS_COLORS = {0: "#4e79a7", 1: "#f28e2b", 2: "#e15759"}
CLASS_MARKERS = {0: "o", 1: "s", 2: "^"}

EMBED_PLOT_DIR = os.path.join(SAVE_DIR, "embedding_diagnostics")
os.makedirs(EMBED_PLOT_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

data = np.load(NPZ_PATH, allow_pickle=True)

X_raw_all     = data["X"].astype(np.float32)
labels_all    = data["labels"]
subjects_all  = data["subjects"]
tasks_all     = data["tasks"]
file_paths_all = data["file_paths"]
start_secs_all = data["start_secs"]

print("Original X:", X_raw_all.shape)
print("Tasks:",    np.unique(tasks_all))
print("Labels:",   np.unique(labels_all))
print("Subjects:", np.unique(subjects_all))

mask = (
    (tasks_all == "Arithmetic") &
    np.isin(labels_all, ["lowlevel", "midlevel", "highlevel"]) &
    (~np.isin(subjects_all, DROP_SUBJECTS))
)

X_raw     = X_raw_all[mask]
labels    = labels_all[mask]
subjects  = subjects_all[mask]
file_paths = file_paths_all[mask]
start_secs = start_secs_all[mask]

y = np.array([LABEL_REMAP[l] for l in labels])

print("\nAfter Arithmetic non-natural + drop subject 7:")
print("X:", X_raw.shape)
print("Subjects:", np.unique(subjects))
print("Label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

# ============================================================
# BALANCE WITHIN EACH SUBJECT
# ============================================================

def balance_within_subjects(X, y, subjects, labels, file_paths, start_secs, seed=42):
    rng = np.random.default_rng(seed)
    keep_indices = []

    print("\n" + "=" * 100)
    print("BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT")
    print("=" * 100)

    for s in sorted(np.unique(subjects)):
        subj_idx = np.where(subjects == s)[0]
        class_counts = {c: int(np.sum(y[subj_idx] == c)) for c in range(N_CLASSES)}
        min_count = min(class_counts.values())
        print(f"Subject {s}: raw counts={class_counts} -> keeping {min_count}/class")

        if min_count == 0:
            print(f"  SKIPPING subject {s}: missing a class")
            continue

        for c in range(N_CLASSES):
            cls_idx = subj_idx[y[subj_idx] == c]
            chosen  = rng.choice(cls_idx, size=min_count, replace=False)
            keep_indices.extend(chosen.tolist())

    keep_indices = np.array(sorted(keep_indices))
    return (
        X[keep_indices], y[keep_indices], subjects[keep_indices],
        labels[keep_indices], file_paths[keep_indices],
        start_secs[keep_indices], keep_indices
    )

X, y, subjects, labels, file_paths, start_secs, kept_original_indices = balance_within_subjects(
    X_raw, y, subjects, labels, file_paths, start_secs, seed=SEED
)

print("\nBalanced X:", X.shape)
print("Balanced global label counts:", {ID_TO_LABEL[i]: int((y == i).sum()) for i in range(3)})

balanced_summary = []
for s in sorted(np.unique(subjects)):
    row = {"subject": s}
    for c in range(N_CLASSES):
        row[ID_TO_LABEL[c]] = int(np.sum((subjects == s) & (y == c)))
    row["total"] = int(np.sum(subjects == s))
    balanced_summary.append(row)

balanced_summary = pd.DataFrame(balanced_summary)
display(balanced_summary)

# ============================================================
# HANDCRAFTED FEATURES
# ============================================================

bands = {
    "theta": (4,  8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 35),
}

def bandpower_from_psd(f, pxx, lo, hi):
    idx = (f >= lo) & (f <= hi)
    return np.trapezoid(pxx[idx], f[idx])

def spectral_entropy_from_psd(pxx):
    p = pxx / (np.sum(pxx) + 1e-12)
    return entropy(p)

def extract_features_one_window(w, fs=250):
    feats = []

    ch_mean = np.mean(w, axis=0);  ch_std = np.std(w, axis=0)
    ch_var  = np.var(w,  axis=0);  ch_ptp = np.ptp(w, axis=0)

    feats.extend(ch_mean.tolist()); feats.extend(ch_std.tolist())
    feats.extend(ch_var.tolist());  feats.extend(ch_ptp.tolist())

    feats.append(np.mean(ch_var)); feats.append(np.max(ch_var))
    feats.append(np.mean(ch_ptp)); feats.append(np.max(ch_ptp))

    band_means = {b: [] for b in bands}
    spec_ents  = []

    for ch in range(w.shape[1]):
        f, pxx = welch(w[:, ch], fs=fs, nperseg=512)
        for name, (lo, hi) in bands.items():
            band_means[name].append(bandpower_from_psd(f, pxx, lo, hi))
        spec_ents.append(spectral_entropy_from_psd(pxx))

    for name in ["theta", "alpha", "beta", "gamma"]:
        vals = np.array(band_means[name])
        feats.append(np.mean(vals))
        feats.append(np.log(np.mean(vals) + 1e-12))
        feats.append(np.std(vals))

    theta = np.mean(band_means["theta"]); alpha = np.mean(band_means["alpha"])
    beta  = np.mean(band_means["beta"]);  gamma = np.mean(band_means["gamma"])

    feats.append(theta / (alpha + 1e-12)); feats.append(theta / (beta + 1e-12))
    feats.append(alpha / (beta  + 1e-12)); feats.append(gamma / (beta + 1e-12))

    feats.append(np.log(theta + 1e-12) - np.log(alpha + 1e-12))
    feats.append(np.log(theta + 1e-12) - np.log(beta  + 1e-12))
    feats.append(np.log(alpha + 1e-12) - np.log(beta  + 1e-12))

    feats.append(np.mean(spec_ents)); feats.append(np.std(spec_ents))

    return np.array(feats, dtype=np.float32)

FEATURE_CACHE = os.path.join(SAVE_DIR, "arithmetic_balanced_baseline_features.npy")

if os.path.exists(FEATURE_CACHE):
    print("\nLoading cached balanced handcrafted features...")
    F_all = np.load(FEATURE_CACHE)
    if len(F_all) != len(X):
        print("Cached feature size mismatch. Recomputing.")
        os.remove(FEATURE_CACHE)
        F_all = None
else:
    F_all = None

if F_all is None:
    print("\nExtracting handcrafted features for BALANCED dataset...")
    feats = []
    for i in range(len(X)):
        if i % 100 == 0:
            print(f"  extracting {i}/{len(X)}")
        feats.append(extract_features_one_window(X[i], FS))
    F_all = np.stack(feats).astype(np.float32)
    np.save(FEATURE_CACHE, F_all)
    print("Saved features:", FEATURE_CACHE)

print("Feature matrix:", F_all.shape)

# ============================================================
# DATASET
# ============================================================

class EEGDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, features=None):
        self.X = X; self.y = y.astype(np.int64); self.features = features

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x     = torch.tensor(self.X[idx].T, dtype=torch.float32)
        label = torch.tensor(self.y[idx],   dtype=torch.long)
        if self.features is None:
            return x, label
        return x, torch.tensor(self.features[idx], dtype=torch.float32), label


def make_loader(X, y, features=None, batch_size=64, shuffle=False, weighted=False):
    ds = EEGDataset(X, y, features)
    if weighted:
        class_counts  = np.bincount(y, minlength=N_CLASSES)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[y]
        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights), replacement=True
        )
        return DataLoader(ds, batch_size=batch_size, sampler=sampler, drop_last=False)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

# ============================================================
# MODELS
# ============================================================

class EEGCNN(nn.Module):
    def __init__(self, n_channels=8, n_classes=3, emb_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32,  kernel_size=15, padding=7), nn.BatchNorm1d(32),  nn.GELU(),
            nn.Conv1d(32,         64,  kernel_size=15, padding=7), nn.BatchNorm1d(64),  nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64,         128, kernel_size=9,  padding=4), nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(128,        128, kernel_size=7,  padding=3), nn.BatchNorm1d(128), nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(nn.Linear(128, emb_dim), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x):
        h = self.encoder(x); h = self.pool(h).squeeze(-1); return self.proj(h)

    def forward(self, x):
        z = self.forward_features(x); return self.classifier(z), z


class EEGCNNFeatureFusion(nn.Module):
    def __init__(self, n_channels=8, n_features=1, n_classes=3, emb_dim=128, feat_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32,  kernel_size=15, padding=7), nn.BatchNorm1d(32),  nn.GELU(),
            nn.Conv1d(32,         64,  kernel_size=15, padding=7), nn.BatchNorm1d(64),  nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64,         128, kernel_size=9,  padding=4), nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(128,        128, kernel_size=7,  padding=3), nn.BatchNorm1d(128), nn.GELU(),
        )
        self.pool     = nn.AdaptiveAvgPool1d(1)
        self.raw_proj = nn.Sequential(nn.Linear(128, emb_dim), nn.GELU(), nn.Dropout(0.2))
        self.feat_proj = nn.Sequential(
            nn.Linear(n_features, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, feat_dim), nn.GELU()
        )
        self.fusion     = nn.Sequential(nn.Linear(emb_dim + feat_dim, emb_dim), nn.GELU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(emb_dim, n_classes)

    def forward_features(self, x, features):
        h      = self.encoder(x); h = self.pool(h).squeeze(-1)
        z_raw  = self.raw_proj(h)
        z_feat = self.feat_proj(features)
        return self.fusion(torch.cat([z_raw, z_feat], dim=1))

    def forward(self, x, features):
        z = self.forward_features(x, features); return self.classifier(z), z

# ============================================================
# EVAL HELPERS
# ============================================================

@torch.no_grad()
def eval_classifier_cnn(model, loader):
    model.eval(); all_preds, all_true = [], []
    for x, yb in loader:
        logits, _ = model(x.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())
    acc  = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm   = confusion_matrix(all_true, all_preds, labels=[0, 1, 2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def eval_classifier_fusion(model, loader):
    model.eval(); all_preds, all_true = [], []
    for x, f, yb in loader:
        logits, _ = model(x.to(device), f.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())
    acc  = accuracy_score(all_true, all_preds)
    bacc = balanced_accuracy_score(all_true, all_preds)
    cm   = confusion_matrix(all_true, all_preds, labels=[0, 1, 2])
    return acc, bacc, cm, np.array(all_true), np.array(all_preds)

@torch.no_grad()
def get_embeddings_cnn(model, loader):
    model.eval(); Z, Y = [], []
    for x, yb in loader:
        _, z = model(x.to(device)); Z.append(z.cpu()); Y.append(yb)
    return torch.cat(Z), torch.cat(Y)

@torch.no_grad()
def get_embeddings_fusion(model, loader):
    model.eval(); Z, Y = [], []
    for x, f, yb in loader:
        _, z = model(x.to(device), f.to(device)); Z.append(z.cpu()); Y.append(yb)
    return torch.cat(Z), torch.cat(Y)

def prototype_eval(Z_support, y_support, Z_query, y_query):
    Z_support = F.normalize(Z_support, dim=1)
    Z_query   = F.normalize(Z_query,   dim=1)
    prototypes = []
    for c in range(N_CLASSES):
        mask  = y_support == c
        proto = Z_support[mask].mean(0) if mask.sum() > 0 else torch.zeros(Z_support.shape[1])
        prototypes.append(F.normalize(proto, dim=0))
    prototypes = torch.stack(prototypes)
    preds      = (Z_query @ prototypes.T).argmax(1)
    acc  = accuracy_score(y_query.numpy(), preds.numpy())
    bacc = balanced_accuracy_score(y_query.numpy(), preds.numpy())
    cm   = confusion_matrix(y_query.numpy(), preds.numpy(), labels=[0, 1, 2])
    return acc, bacc, cm, preds.numpy()

# ============================================================
# SPLITTING / BASELINE
# ============================================================

def make_fewshot_split(test_indices, y, k_per_class=5, seed=42):
    rng = np.random.default_rng(seed)
    support_idx, query_idx = [], []
    print("  Few-shot split:")
    for c in range(N_CLASSES):
        cls_idx = test_indices[y[test_indices] == c]
        rng.shuffle(cls_idx)
        k = min(k_per_class, len(cls_idx) // 2)
        support_idx.extend(cls_idx[:k])
        query_idx.extend(cls_idx[k:])
        pct = 100 * k / len(cls_idx) if len(cls_idx) > 0 else 0
        print(f"    class {c} ({ID_TO_LABEL[c]}): total={len(cls_idx)}, "
              f"support={k}, query={len(cls_idx)-k}, support%={pct:.1f}%")
    return np.array(support_idx), np.array(query_idx)


def majority_baseline(y_query):
    vals, counts = np.unique(y_query, return_counts=True)
    majority = vals[np.argmax(counts)]
    preds    = np.full_like(y_query, fill_value=majority)
    return (accuracy_score(y_query, preds),
            balanced_accuracy_score(y_query, preds),
            confusion_matrix(y_query, preds, labels=[0, 1, 2]))

# ============================================================
# ============================================================
#   INTRA / INTER CLASS DISTANCE UTILITIES  (NEW)
# ============================================================
# ============================================================

def compute_intra_inter_distances(Z_t, y_t, normalize=True):
    """
    Compute mean intra-class and inter-class L2 distances in embedding space.

    Parameters
    ----------
    Z_t  : torch.Tensor  [N, D]
    y_t  : torch.Tensor  [N]   integer class labels
    normalize : bool  – L2-normalise embeddings first (operates on cosine geometry)

    Returns
    -------
    dict with keys:
        intra_mean        – mean over all classes of mean within-class pairwise dist
        inter_mean        – mean over all class-pairs of mean between-class pairwise dist
        ratio             – inter_mean / (intra_mean + 1e-8)   (higher = better separated)
        intra_per_class   – {class_id: float}
        inter_per_pair    – {(ci, cj): float}  ci < cj
        class_centroids   – {class_id: np.ndarray [D]}
        centroid_distances– {(ci, cj): float}  Euclidean dist between class centroids
    """
    if normalize:
        Z_t = F.normalize(Z_t.float(), dim=1)
    Z_np = Z_t.numpy()
    y_np = y_t.numpy() if isinstance(y_t, torch.Tensor) else y_t

    intra_per_class    = {}
    centroids          = {}

    for c in range(N_CLASSES):
        idx = np.where(y_np == c)[0]
        if len(idx) < 2:
            intra_per_class[c] = float("nan")
            centroids[c]       = Z_np[idx[0]] if len(idx) == 1 else np.zeros(Z_np.shape[1])
            continue
        Zc = Z_np[idx]
        centroids[c] = Zc.mean(axis=0)
        # pairwise L2 via broadcasting  (memory-safe for typical fold sizes)
        diff = Zc[:, None, :] - Zc[None, :, :]       # [N, N, D]
        dists = np.linalg.norm(diff, axis=-1)          # [N, N]
        # upper triangle only (exclude diagonal)
        n = len(idx)
        iu = np.triu_indices(n, k=1)
        intra_per_class[c] = float(dists[iu].mean()) if len(iu[0]) > 0 else 0.0

    inter_per_pair     = {}
    centroid_distances = {}

    for ci in range(N_CLASSES):
        for cj in range(ci + 1, N_CLASSES):
            idx_i = np.where(y_np == ci)[0]
            idx_j = np.where(y_np == cj)[0]
            if len(idx_i) == 0 or len(idx_j) == 0:
                inter_per_pair[(ci, cj)]     = float("nan")
                centroid_distances[(ci, cj)] = float("nan")
                continue
            Zi = Z_np[idx_i]; Zj = Z_np[idx_j]
            # cross-pairwise distances
            diff  = Zi[:, None, :] - Zj[None, :, :]
            dists = np.linalg.norm(diff, axis=-1)
            inter_per_pair[(ci, cj)]     = float(dists.mean())
            centroid_distances[(ci, cj)] = float(
                np.linalg.norm(centroids[ci] - centroids[cj])
            )

    valid_intra = [v for v in intra_per_class.values() if not np.isnan(v)]
    valid_inter = [v for v in inter_per_pair.values()  if not np.isnan(v)]

    intra_mean = float(np.mean(valid_intra)) if valid_intra else float("nan")
    inter_mean = float(np.mean(valid_inter)) if valid_inter else float("nan")
    ratio      = inter_mean / (intra_mean + 1e-8)

    return {
        "intra_mean":         intra_mean,
        "inter_mean":         inter_mean,
        "ratio":              ratio,
        "intra_per_class":    intra_per_class,
        "inter_per_pair":     inter_per_pair,
        "class_centroids":    centroids,
        "centroid_distances": centroid_distances,
    }


def print_distance_report(dist_dict, split_name, subject, model_name):
    """Pretty-print the intra/inter distance report."""
    print(f"\n{'─'*80}")
    print(f"INTRA/INTER DISTANCES | {model_name} | Subject {subject} | {split_name}")
    print(f"{'─'*80}")
    print(f"  Intra-class mean  : {dist_dict['intra_mean']:.4f}")
    print(f"  Inter-class mean  : {dist_dict['inter_mean']:.4f}")
    print(f"  Ratio (inter/intra): {dist_dict['ratio']:.4f}  "
          f"({'good separation' if dist_dict['ratio'] > 1 else 'classes overlap'})")
    print("  Per-class intra distances:")
    for c, v in dist_dict["intra_per_class"].items():
        print(f"    {ID_TO_LABEL[c]:<12}: {v:.4f}")
    print("  Per-pair inter distances:")
    for (ci, cj), v in dist_dict["inter_per_pair"].items():
        print(f"    {ID_TO_LABEL[ci]} vs {ID_TO_LABEL[cj]:<12}: {v:.4f}   "
              f"centroid dist={dist_dict['centroid_distances'][(ci,cj)]:.4f}")


def plot_distance_heatmaps(dist_dict, split_name, subject, model_name):
    """
    Two subplots:
      Left  – N_CLASSES × N_CLASSES symmetric matrix of mean pairwise distances
              (diagonal = intra, off-diagonal = inter)
      Right – bar chart: intra per class + inter per pair
    """
    n = N_CLASSES
    mat = np.full((n, n), np.nan)
    for c, v in dist_dict["intra_per_class"].items():
        mat[c, c] = v
    for (ci, cj), v in dist_dict["inter_per_pair"].items():
        mat[ci, cj] = v; mat[cj, ci] = v

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(f"{model_name}  |  Subject {subject}  |  {split_name}",
                 fontsize=11, fontweight="bold")

    # --- heatmap ---
    ax = axes[0]
    masked = np.ma.masked_invalid(mat)
    im = ax.imshow(masked, cmap="RdYlGn_r", aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8)
    tick_labels = [ID_TO_LABEL[c] for c in range(n)]
    ax.set_xticks(range(n)); ax.set_xticklabels(tick_labels, fontsize=8)
    ax.set_yticks(range(n)); ax.set_yticklabels(tick_labels, fontsize=8)
    ax.set_title("Pairwise distance matrix\n(diag=intra, off-diag=inter)", fontsize=9)
    for i in range(n):
        for j in range(n):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center",
                        fontsize=8, color="black")

    # --- bar chart ---
    ax2 = axes[1]
    bar_labels, bar_vals, bar_colors = [], [], []
    for c in range(n):
        bar_labels.append(f"intra\n{ID_TO_LABEL[c]}")
        bar_vals.append(dist_dict["intra_per_class"][c] if not np.isnan(dist_dict["intra_per_class"][c]) else 0)
        bar_colors.append(CLASS_COLORS[c])
    for (ci, cj), v in dist_dict["inter_per_pair"].items():
        bar_labels.append(f"inter\n{ID_TO_LABEL[ci][:3]}-{ID_TO_LABEL[cj][:3]}")
        bar_vals.append(v if not np.isnan(v) else 0)
        bar_colors.append("#aaaaaa")

    xs = np.arange(len(bar_labels))
    ax2.bar(xs, bar_vals, color=bar_colors, edgecolor="white", linewidth=0.5)
    ax2.set_xticks(xs); ax2.set_xticklabels(bar_labels, fontsize=7)
    ax2.set_ylabel("Mean L2 distance (normalised emb.)", fontsize=8)
    ax2.set_title(f"Intra vs Inter distances\nratio={dist_dict['ratio']:.3f}", fontsize=9)
    ax2.axhline(dist_dict["intra_mean"], color="steelblue", linestyle="--",
                linewidth=1, label=f"intra_mean={dist_dict['intra_mean']:.3f}")
    ax2.axhline(dist_dict["inter_mean"], color="tomato", linestyle="--",
                linewidth=1, label=f"inter_mean={dist_dict['inter_mean']:.3f}")
    ax2.legend(fontsize=7)

    plt.tight_layout()
    fname = f"{model_name}_subj{subject}_{split_name.replace(' ','_')}_dist_heatmap"
    save_fig(fig, fname)


def plot_distance_summary_across_subjects(all_dist_records, model_name):
    """
    Summarise intra/inter distances across all LOSO folds.
    all_dist_records: list of dicts with keys
        subject, split, intra_mean, inter_mean, ratio,
        intra_low, intra_mid, intra_high,
        inter_low_mid, inter_low_high, inter_mid_high
    """
    df = pd.DataFrame(all_dist_records)
    splits = df["split"].unique()

    for split in splits:
        sdf = df[df["split"] == split].copy()
        subjects_ord = sdf["subject"].tolist()
        x = np.arange(len(subjects_ord))
        width = 0.28

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"{model_name}  —  Intra/Inter distances across subjects  [{split}]",
                     fontsize=11, fontweight="bold")

        # --- subplot 1: intra per class ---
        ax = axes[0]
        for ci, label_key in enumerate(["intra_low", "intra_mid", "intra_high"]):
            ax.bar(x + (ci - 1) * width, sdf[label_key], width,
                   label=ID_TO_LABEL[ci], color=CLASS_COLORS[ci], alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels([f"S{s}" for s in subjects_ord], fontsize=8)
        ax.set_ylabel("Mean intra-class distance"); ax.set_title("Per-class intra distances")
        ax.legend(fontsize=8)

        # --- subplot 2: inter per pair + ratio ---
        ax2 = axes[1]
        pair_cols   = ["inter_low_mid", "inter_low_high", "inter_mid_high"]
        pair_labels = ["low-mid", "low-high", "mid-high"]
        pair_colors = ["#9c9c9c", "#636363", "#333333"]
        for pi, (col, lbl, col_c) in enumerate(zip(pair_cols, pair_labels, pair_colors)):
            ax2.bar(x + (pi - 1) * width, sdf[col], width,
                    label=lbl, color=col_c, alpha=0.8)
        ax2.set_xticks(x); ax2.set_xticklabels([f"S{s}" for s in subjects_ord], fontsize=8)
        ax2.set_ylabel("Mean inter-class distance"); ax2.set_title("Per-pair inter distances")
        ax2.legend(fontsize=8)

        # overlay ratio on twin axis
        ax3 = ax2.twinx()
        ax3.plot(x, sdf["ratio"], color="crimson", marker="D",
                 markersize=6, linewidth=1.5, label="ratio (inter/intra)")
        ax3.axhline(1.0, color="crimson", linestyle=":", linewidth=0.8)
        ax3.set_ylabel("inter/intra ratio", color="crimson", fontsize=9)
        ax3.tick_params(axis="y", colors="crimson")
        ax3.legend(loc="upper right", fontsize=7)

        plt.tight_layout()
        save_fig(fig, f"{model_name}_ALL_SUBJECTS_{split.replace(' ','_')}_dist_summary")

    # Also print a summary table
    print(f"\n{'='*100}")
    print(f"DISTANCE SUMMARY TABLE — {model_name}")
    print(f"{'='*100}")
    display(df)
    csv_path = os.path.join(SAVE_DIR, f"{model_name}_distance_summary.csv")
    df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)


def _dist_record(dist_dict, subject, split):
    """Flatten a dist_dict into a single row dict for the summary DataFrame."""
    return {
        "subject":        subject,
        "split":          split,
        "intra_mean":     dist_dict["intra_mean"],
        "inter_mean":     dist_dict["inter_mean"],
        "ratio":          dist_dict["ratio"],
        "intra_low":      dist_dict["intra_per_class"].get(0, float("nan")),
        "intra_mid":      dist_dict["intra_per_class"].get(1, float("nan")),
        "intra_high":     dist_dict["intra_per_class"].get(2, float("nan")),
        "inter_low_mid":  dist_dict["inter_per_pair"].get((0, 1), float("nan")),
        "inter_low_high": dist_dict["inter_per_pair"].get((0, 2), float("nan")),
        "inter_mid_high": dist_dict["inter_per_pair"].get((1, 2), float("nan")),
    }


# ============================================================
# ============================================================
#   EMBEDDING VISUALIZATION UTILITIES  (unchanged from original)
# ============================================================
# ============================================================

def _fit_pca_tsne(Z_np, seed=SEED):
    """Return (Z_pca, Z_tsne, pca_var) after standard scaling."""
    Z_sc = StandardScaler().fit_transform(Z_np)

    pca   = PCA(n_components=2, random_state=seed)
    Z_pca = pca.fit_transform(Z_sc)

    n_pts  = len(Z_sc)
    perp   = max(3, min(30, (n_pts - 1) // 3))
    tsne   = TSNE(n_components=2, perplexity=perp, init="pca",
                  learning_rate="auto", random_state=seed)
    Z_tsne = tsne.fit_transform(Z_sc)

    return Z_pca, Z_tsne, pca.explained_variance_ratio_.sum(), perp


def _scatter_2d(ax, Z_2d, Y, title, xlabel, ylabel, show_legend=True):
    for c in range(N_CLASSES):
        idx = Y == c
        ax.scatter(Z_2d[idx, 0], Z_2d[idx, 1],
                   s=40, alpha=0.75, label=ID_TO_LABEL[c],
                   color=CLASS_COLORS[c], marker=CLASS_MARKERS[c])
    ax.set_title(title, fontsize=9, pad=4)
    ax.set_xlabel(xlabel, fontsize=8); ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(labelsize=7)
    if show_legend:
        ax.legend(fontsize=7, markerscale=1.2)


def save_fig(fig, fname):
    path = os.path.join(EMBED_PLOT_DIR, fname + ".png")
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved:", path)
    plt.close(fig)


def plot_fold_embeddings(Z_train_np, y_train_np,
                         Z_query_np, y_query_np,
                         subject, model_name, epoch):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    fig.suptitle(f"{model_name}  |  Subject {subject} held-out  |  Epoch {epoch}",
                 fontsize=11, fontweight="bold")

    for row, (Z_np, Y_np, split_name) in enumerate([
        (Z_train_np, y_train_np, "Train"),
        (Z_query_np, y_query_np, "Query (held-out)"),
    ]):
        Z_pca, Z_tsne, var, perp = _fit_pca_tsne(Z_np)

        _scatter_2d(axes[row, 0], Z_pca, Y_np,
                    title=f"{split_name} — PCA  (var={var:.3f})",
                    xlabel="PC1", ylabel="PC2")
        _scatter_2d(axes[row, 1], Z_tsne, Y_np,
                    title=f"{split_name} — t-SNE  (perp={perp})",
                    xlabel="t-SNE 1", ylabel="t-SNE 2")

    plt.tight_layout()
    save_fig(fig, f"{model_name}_subj{subject}_ep{epoch:02d}_train_query_embed")


def plot_support_query_overlay(Z_support_np, y_support_np,
                               Z_query_np,   y_query_np,
                               subject, model_name, epoch):
    Z_all = np.concatenate([Z_support_np, Z_query_np], axis=0)
    y_all = np.concatenate([y_support_np, y_query_np], axis=0)
    split = np.array(["support"] * len(Z_support_np) + ["query"] * len(Z_query_np))

    Z_sc  = StandardScaler().fit_transform(Z_all)
    pca   = PCA(n_components=2, random_state=SEED)
    Z_pca = pca.fit_transform(Z_sc)

    fig, ax = plt.subplots(figsize=(6, 5))
    for c in range(N_CLASSES):
        for sp, marker, sz, alpha in [("support", "X", 120, 0.95),
                                       ("query",   "o",  40, 0.55)]:
            idx = (y_all == c) & (split == sp)
            ax.scatter(Z_pca[idx, 0], Z_pca[idx, 1],
                       s=sz, alpha=alpha, marker=marker,
                       color=CLASS_COLORS[c],
                       label=f"{ID_TO_LABEL[c]} ({sp})" if c == 0 or sp == "query" else "")

    from matplotlib.lines import Line2D
    handles = []
    for c in range(N_CLASSES):
        handles.append(Line2D([0], [0], marker="X", color="w",
                               markerfacecolor=CLASS_COLORS[c], markersize=9,
                               label=f"{ID_TO_LABEL[c]} support"))
        handles.append(Line2D([0], [0], marker="o", color="w",
                               markerfacecolor=CLASS_COLORS[c], markersize=7, alpha=0.7,
                               label=f"{ID_TO_LABEL[c]} query"))
    ax.legend(handles=handles, fontsize=7, ncol=2)

    ax.set_title(f"{model_name}  |  Subject {subject}  |  Epoch {epoch}\n"
                 f"Support (✕) vs Query (●) — PCA  (var={pca.explained_variance_ratio_.sum():.3f})",
                 fontsize=9)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    plt.tight_layout()
    save_fig(fig, f"{model_name}_subj{subject}_ep{epoch:02d}_support_query_overlay")


def plot_all_subjects_grid(all_subject_embeddings, model_name):
    n_subj = len(all_subject_embeddings)
    if n_subj == 0:
        return

    fig, axes = plt.subplots(n_subj, 2,
                              figsize=(11, 4.5 * n_subj),
                              squeeze=False)
    fig.suptitle(f"{model_name}  —  All Subjects  (query split, final epoch)",
                 fontsize=13, fontweight="bold", y=1.002)

    for row, entry in enumerate(all_subject_embeddings):
        subj     = entry["subject"]
        Z_np     = entry["Z"]
        Y_np     = entry["Y"]
        fs_acc   = entry["fs_acc"]
        zs_acc   = entry["zs_acc"]

        Z_pca, Z_tsne, var, perp = _fit_pca_tsne(Z_np)

        _scatter_2d(axes[row, 0], Z_pca, Y_np,
                    title=(f"Subj {subj} — PCA (var={var:.3f}) | "
                           f"ZS={zs_acc:.3f} FS={fs_acc:.3f}"),
                    xlabel="PC1", ylabel="PC2",
                    show_legend=(row == 0))

        _scatter_2d(axes[row, 1], Z_tsne, Y_np,
                    title=(f"Subj {subj} — t-SNE (perp={perp}) | "
                           f"ZS={zs_acc:.3f} FS={fs_acc:.3f}"),
                    xlabel="t-SNE 1", ylabel="t-SNE 2",
                    show_legend=False)

    handles = [plt.Line2D([0], [0], marker=CLASS_MARKERS[c], color="w",
                           markerfacecolor=CLASS_COLORS[c], markersize=9,
                           label=ID_TO_LABEL[c]) for c in range(N_CLASSES)]
    fig.legend(handles=handles, loc="upper right", fontsize=9,
               bbox_to_anchor=(1.0, 1.0), frameon=True)

    plt.tight_layout()
    save_fig(fig, f"{model_name}_ALL_SUBJECTS_query_embed_grid")


def prototype_distance_report(Z_support_t, y_support_t,
                               Z_query_t,   y_query_t,
                               subject, model_name, epoch):
    Zs = F.normalize(Z_support_t.float(), dim=1)
    Zq = F.normalize(Z_query_t.float(),   dim=1)
    protos = []
    for c in range(N_CLASSES):
        m     = y_support_t == c
        proto = Zs[m].mean(0) if m.sum() > 0 else torch.zeros(Zs.shape[1])
        protos.append(F.normalize(proto, dim=0))
    protos = torch.stack(protos)
    sims   = Zq @ protos.T
    preds  = sims.argmax(1)

    print(f"\n{'─'*80}")
    print(f"PROTOTYPE DISTANCES | {model_name} | Subject {subject} | Epoch {epoch}")
    print(f"{'─'*80}")
    for c in range(N_CLASSES):
        idx  = y_query_t == c
        if idx.sum() == 0: continue
        own  = sims[idx, c]
        corr = (preds[idx] == c)
        print(f"  {ID_TO_LABEL[c]:<10}  n={idx.sum().item():3d}  "
              f"sim_own={own.mean().item():.4f}  "
              f"correct={corr.sum().item()}/{idx.sum().item()}")
    print(f"  Mean max-sim: {sims.max(1).values.mean().item():.4f}")
    print(f"  Pred counts: {np.bincount(preds.numpy(), minlength=N_CLASSES)}")


# ============================================================
# CNN ONLY LOSO
# ============================================================

def run_loso_cnn_only():
    print("\n" + "=" * 100)
    print("RUNNING BALANCED BASELINE 1: CNN ONLY")
    print("=" * 100)

    fold_results            = []
    all_subject_embeddings  = []
    all_dist_records        = []          # NEW: for distance summary
    unique_subjects         = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#" * 100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#" * 100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx  = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx]  == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED + fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query"); continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)
        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        X_train_raw   = X[train_idx];   X_support_raw = X[support_idx]; X_query_raw = X[query_idx]
        train_mean    = X_train_raw.mean(axis=(0, 1), keepdims=True)
        train_std     = X_train_raw.std(axis=(0,  1), keepdims=True) + 1e-6

        X_train   = (X_train_raw   - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query   = (X_query_raw   - train_mean) / train_std

        y_train   = y[train_idx];  y_support = y[support_idx]; y_query = y[query_idx]

        train_loader       = make_loader(X_train,   y_train,   batch_size=BATCH_SIZE, weighted=True)
        support_loader     = make_loader(X_support, y_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader       = make_loader(X_query,   y_query,   batch_size=BATCH_SIZE, shuffle=False)
        train_embed_loader = make_loader(X_train,   y_train,   batch_size=BATCH_SIZE, shuffle=False)

        model     = EEGCNN(n_channels=8, n_classes=3, emb_dim=128).to(device)
        opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = best_fs_bacc = best_zs_acc = best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0; all_preds, all_true = [], []

            for xb, yb in train_loader:
                xb = xb.to(device); yb = yb.to(device)
                opt.zero_grad()
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward(); opt.step()
                total_loss += loss.item() * xb.size(0)
                all_preds.extend(logits.argmax(1).detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss  = total_loss / len(train_loader.dataset)
            train_acc   = accuracy_score(all_true, all_preds)
            train_bacc  = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_cnn(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_cnn(model, support_loader)
            Z_q,   y_q_t  = get_embeddings_cnn(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc  = max(best_fs_acc,  fs_acc);  best_fs_bacc  = max(best_fs_bacc,  fs_bacc)
            best_zs_acc  = max(best_zs_acc,  zs_acc);  best_zs_bacc  = max(best_zs_bacc,  zs_bacc)

            print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | "
                  f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                  f"zero_shot={zs_acc:.4f} | zs_bacc={zs_bacc:.4f} | "
                  f"fewshot={fs_acc:.4f} | fs_bacc={fs_bacc:.4f} | "
                  f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}")
            print("  Zero-shot CM:"); print(zs_cm)
            print("  Few-shot prototype CM:"); print(fs_cm)

        # ---- EMBEDDING VISUALIZATIONS (final epoch) ----
        print(f"\n[VIZ] Generating embedding plots for subject {test_subj}...")

        Z_train_np = get_embeddings_cnn(model, train_embed_loader)[0].numpy()
        y_train_np = y_train
        Z_q_np     = Z_q.numpy()
        y_q_np     = y_q_t.numpy()
        Z_sup_np   = Z_sup.numpy()
        y_sup_np   = y_sup_t.numpy()

        plot_fold_embeddings(Z_train_np, y_train_np,
                             Z_q_np,    y_q_np,
                             subject=test_subj, model_name="CNN_only", epoch=EPOCHS)

        plot_support_query_overlay(Z_sup_np, y_sup_np,
                                   Z_q_np,   y_q_np,
                                   subject=test_subj, model_name="CNN_only", epoch=EPOCHS)

        prototype_distance_report(Z_sup, y_sup_t, Z_q, y_q_t,
                                  subject=test_subj, model_name="CNN_only", epoch=EPOCHS)

        # ---- NEW: INTRA/INTER DISTANCE ANALYSIS ----
        print(f"\n[DIST] Computing intra/inter distances for subject {test_subj}...")

        # Train split distances
        Z_train_t = torch.tensor(Z_train_np)
        y_train_t = torch.tensor(y_train_np)
        dist_train = compute_intra_inter_distances(Z_train_t, y_train_t)
        print_distance_report(dist_train, "Train split", test_subj, "CNN_only")
        plot_distance_heatmaps(dist_train, "Train split", test_subj, "CNN_only")
        all_dist_records.append(_dist_record(dist_train, test_subj, "train"))

        # Query split distances
        dist_query = compute_intra_inter_distances(Z_q, y_q_t)
        print_distance_report(dist_query, "Query split", test_subj, "CNN_only")
        plot_distance_heatmaps(dist_query, "Query split", test_subj, "CNN_only")
        all_dist_records.append(_dist_record(dist_query, test_subj, "query"))

        # Support split distances
        dist_support = compute_intra_inter_distances(Z_sup, y_sup_t)
        print_distance_report(dist_support, "Support split", test_subj, "CNN_only")
        all_dist_records.append(_dist_record(dist_support, test_subj, "support"))

        # Collect for all-subjects grid
        all_subject_embeddings.append({
            "subject": test_subj,
            "Z": Z_q_np,
            "Y": y_q_np,
            "fs_acc": best_fs_acc,
            "zs_acc": best_zs_acc,
        })

        print("\nFINAL FOLD RESULT")
        print(f"Subject {test_subj} | best ZS={best_zs_acc:.4f} | ZS_bacc={best_zs_bacc:.4f} | "
              f"FS={best_fs_acc:.4f} | FS_bacc={best_fs_bacc:.4f}")

        fold_results.append({
            "subject":              test_subj,
            "best_zero_shot":       best_zs_acc,
            "best_zero_shot_bacc":  best_zs_bacc,
            "best_fewshot":         best_fs_acc,
            "best_fewshot_bacc":    best_fs_bacc,
            "majority_acc":         maj_acc,
            "majority_bacc":        maj_bacc,
            "n_train":              len(train_idx),
            "n_support":            len(support_idx),
            "n_query":              len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx),
            # distance metrics for the query split
            "query_intra_mean":     dist_query["intra_mean"],
            "query_inter_mean":     dist_query["inter_mean"],
            "query_dist_ratio":     dist_query["ratio"],
        })

    # ---- ALL-SUBJECTS GRID ----
    print("\n[VIZ] Generating all-subjects embedding grid...")
    plot_all_subjects_grid(all_subject_embeddings, model_name="CNN_only")

    # ---- NEW: CROSS-SUBJECT DISTANCE SUMMARY ----
    print("\n[DIST] Generating cross-subject distance summary plots...")
    plot_distance_summary_across_subjects(all_dist_records, model_name="CNN_only")

    results_df = pd.DataFrame(fold_results)

    print("\n" + "=" * 100)
    print("BALANCED CNN ONLY LOSO SUMMARY")
    print("=" * 100)
    display(results_df)
    print("Mean ZS:              ", results_df["best_zero_shot"].mean())
    print("Mean ZS bacc:         ", results_df["best_zero_shot_bacc"].mean())
    print("Mean FS:              ", results_df["best_fewshot"].mean())
    print("Mean FS bacc:         ", results_df["best_fewshot_bacc"].mean())
    print("Mean majority:        ", results_df["majority_acc"].mean())
    print("Mean support %:       ", results_df["support_percent_total"].mean())
    print("Mean query intra dist:", results_df["query_intra_mean"].mean())
    print("Mean query inter dist:", results_df["query_inter_mean"].mean())
    print("Mean query dist ratio:", results_df["query_dist_ratio"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_only_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# CNN + FEATURE FUSION LOSO
# ============================================================

def run_loso_cnn_feature_fusion():
    print("\n" + "=" * 100)
    print("RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION")
    print("=" * 100)

    fold_results           = []
    all_subject_embeddings = []
    all_dist_records       = []          # NEW
    unique_subjects        = sorted(np.unique(subjects))

    for fold, test_subj in enumerate(unique_subjects, 1):
        print("\n" + "#" * 100)
        print(f"FOLD {fold}/{len(unique_subjects)} | HELD OUT SUBJECT = {test_subj}")
        print("#" * 100)

        train_idx = np.where(subjects != test_subj)[0]
        test_idx  = np.where(subjects == test_subj)[0]

        print("Train windows:", len(train_idx))
        print("Held-out windows:", len(test_idx))
        print("Train label counts:", {ID_TO_LABEL[i]: int((y[train_idx] == i).sum()) for i in range(3)})
        print("Held label counts :", {ID_TO_LABEL[i]: int((y[test_idx]  == i).sum()) for i in range(3)})

        support_idx, query_idx = make_fewshot_split(test_idx, y, FEW_SHOT_PER_CLASS, seed=SEED + fold)

        if len(query_idx) == 0:
            print("SKIPPING fold: empty query"); continue

        y_query_raw = y[query_idx]
        maj_acc, maj_bacc, maj_cm = majority_baseline(y_query_raw)
        print(f"Majority baseline query acc={maj_acc:.4f} | balanced_acc={maj_bacc:.4f}")
        print(maj_cm)

        X_train_raw   = X[train_idx];   X_support_raw = X[support_idx]; X_query_raw = X[query_idx]
        train_mean    = X_train_raw.mean(axis=(0, 1), keepdims=True)
        train_std     = X_train_raw.std(axis=(0,  1), keepdims=True) + 1e-6
        X_train   = (X_train_raw   - train_mean) / train_std
        X_support = (X_support_raw - train_mean) / train_std
        X_query   = (X_query_raw   - train_mean) / train_std

        F_train_raw   = F_all[train_idx]; F_support_raw = F_all[support_idx]; F_query_raw = F_all[query_idx]
        feat_mean     = F_train_raw.mean(axis=0, keepdims=True)
        feat_std      = F_train_raw.std(axis=0,  keepdims=True) + 1e-6
        F_train   = (F_train_raw   - feat_mean) / feat_std
        F_support = (F_support_raw - feat_mean) / feat_std
        F_query   = (F_query_raw   - feat_mean) / feat_std

        y_train = y[train_idx]; y_support = y[support_idx]; y_query = y[query_idx]

        train_loader        = make_loader(X_train,   y_train,   features=F_train,   batch_size=BATCH_SIZE, weighted=True)
        support_loader      = make_loader(X_support, y_support, features=F_support, batch_size=BATCH_SIZE, shuffle=False)
        query_loader        = make_loader(X_query,   y_query,   features=F_query,   batch_size=BATCH_SIZE, shuffle=False)
        train_embed_loader  = make_loader(X_train,   y_train,   features=F_train,   batch_size=BATCH_SIZE, shuffle=False)

        model = EEGCNNFeatureFusion(
            n_channels=8, n_features=F_all.shape[1],
            n_classes=3,  emb_dim=128, feat_dim=64
        ).to(device)

        opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()

        best_fs_acc = best_fs_bacc = best_zs_acc = best_zs_bacc = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0; all_preds, all_true = [], []

            for xb, fb, yb in train_loader:
                xb = xb.to(device); fb = fb.to(device); yb = yb.to(device)
                opt.zero_grad()
                logits, _ = model(xb, fb)
                loss = criterion(logits, yb)
                loss.backward(); opt.step()
                total_loss += loss.item() * xb.size(0)
                all_preds.extend(logits.argmax(1).detach().cpu().numpy())
                all_true.extend(yb.detach().cpu().numpy())

            train_loss = total_loss / len(train_loader.dataset)
            train_acc  = accuracy_score(all_true, all_preds)
            train_bacc = balanced_accuracy_score(all_true, all_preds)

            zs_acc, zs_bacc, zs_cm, _, _ = eval_classifier_fusion(model, query_loader)

            Z_sup, y_sup_t = get_embeddings_fusion(model, support_loader)
            Z_q,   y_q_t  = get_embeddings_fusion(model, query_loader)
            fs_acc, fs_bacc, fs_cm, _ = prototype_eval(Z_sup, y_sup_t, Z_q, y_q_t)

            best_fs_acc  = max(best_fs_acc,  fs_acc);  best_fs_bacc  = max(best_fs_bacc,  fs_bacc)
            best_zs_acc  = max(best_zs_acc,  zs_acc);  best_zs_bacc  = max(best_zs_bacc,  zs_bacc)

            print(f"Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | "
                  f"train_acc={train_acc:.4f} | train_bacc={train_bacc:.4f} | "
                  f"zero_shot={zs_acc:.4f} | zs_bacc={zs_bacc:.4f} | "
                  f"fewshot={fs_acc:.4f} | fs_bacc={fs_bacc:.4f} | "
                  f"best_zs={best_zs_acc:.4f} | best_fs={best_fs_acc:.4f}")
            print("  Zero-shot CM:"); print(zs_cm)
            print("  Few-shot prototype CM:"); print(fs_cm)

        # ---- EMBEDDING VISUALIZATIONS (final epoch) ----
        print(f"\n[VIZ] Generating embedding plots for subject {test_subj}...")

        Z_train_np = get_embeddings_fusion(model, train_embed_loader)[0].numpy()
        y_train_np = y_train
        Z_q_np     = Z_q.numpy()
        y_q_np     = y_q_t.numpy()
        Z_sup_np   = Z_sup.numpy()
        y_sup_np   = y_sup_t.numpy()

        plot_fold_embeddings(Z_train_np, y_train_np,
                             Z_q_np,    y_q_np,
                             subject=test_subj, model_name="CNN_fusion", epoch=EPOCHS)

        plot_support_query_overlay(Z_sup_np, y_sup_np,
                                   Z_q_np,   y_q_np,
                                   subject=test_subj, model_name="CNN_fusion", epoch=EPOCHS)

        prototype_distance_report(Z_sup, y_sup_t, Z_q, y_q_t,
                                  subject=test_subj, model_name="CNN_fusion", epoch=EPOCHS)

        # ---- NEW: INTRA/INTER DISTANCE ANALYSIS ----
        print(f"\n[DIST] Computing intra/inter distances for subject {test_subj}...")

        Z_train_t = torch.tensor(Z_train_np)
        y_train_t = torch.tensor(y_train_np)
        dist_train = compute_intra_inter_distances(Z_train_t, y_train_t)
        print_distance_report(dist_train, "Train split", test_subj, "CNN_fusion")
        plot_distance_heatmaps(dist_train, "Train split", test_subj, "CNN_fusion")
        all_dist_records.append(_dist_record(dist_train, test_subj, "train"))

        dist_query = compute_intra_inter_distances(Z_q, y_q_t)
        print_distance_report(dist_query, "Query split", test_subj, "CNN_fusion")
        plot_distance_heatmaps(dist_query, "Query split", test_subj, "CNN_fusion")
        all_dist_records.append(_dist_record(dist_query, test_subj, "query"))

        dist_support = compute_intra_inter_distances(Z_sup, y_sup_t)
        print_distance_report(dist_support, "Support split", test_subj, "CNN_fusion")
        all_dist_records.append(_dist_record(dist_support, test_subj, "support"))

        all_subject_embeddings.append({
            "subject": test_subj,
            "Z": Z_q_np,
            "Y": y_q_np,
            "fs_acc": best_fs_acc,
            "zs_acc": best_zs_acc,
        })

        print("\nFINAL FOLD RESULT")
        print(f"Subject {test_subj} | best ZS={best_zs_acc:.4f} | ZS_bacc={best_zs_bacc:.4f} | "
              f"FS={best_fs_acc:.4f} | FS_bacc={best_fs_bacc:.4f}")

        fold_results.append({
            "subject":               test_subj,
            "best_zero_shot":        best_zs_acc,
            "best_zero_shot_bacc":   best_zs_bacc,
            "best_fewshot":          best_fs_acc,
            "best_fewshot_bacc":     best_fs_bacc,
            "majority_acc":          maj_acc,
            "majority_bacc":         maj_bacc,
            "n_train":               len(train_idx),
            "n_support":             len(support_idx),
            "n_query":               len(query_idx),
            "support_percent_total": 100 * len(support_idx) / len(test_idx),
            # distance metrics for the query split
            "query_intra_mean":      dist_query["intra_mean"],
            "query_inter_mean":      dist_query["inter_mean"],
            "query_dist_ratio":      dist_query["ratio"],
        })

    # ---- ALL-SUBJECTS GRID ----
    print("\n[VIZ] Generating all-subjects embedding grid...")
    plot_all_subjects_grid(all_subject_embeddings, model_name="CNN_fusion")

    # ---- NEW: CROSS-SUBJECT DISTANCE SUMMARY ----
    print("\n[DIST] Generating cross-subject distance summary plots...")
    plot_distance_summary_across_subjects(all_dist_records, model_name="CNN_fusion")

    results_df = pd.DataFrame(fold_results)

    print("\n" + "=" * 100)
    print("BALANCED CNN + FEATURE FUSION LOSO SUMMARY")
    print("=" * 100)
    display(results_df)
    print("Mean ZS:              ", results_df["best_zero_shot"].mean())
    print("Mean ZS bacc:         ", results_df["best_zero_shot_bacc"].mean())
    print("Mean FS:              ", results_df["best_fewshot"].mean())
    print("Mean FS bacc:         ", results_df["best_fewshot_bacc"].mean())
    print("Mean majority:        ", results_df["majority_acc"].mean())
    print("Mean support %:       ", results_df["support_percent_total"].mean())
    print("Mean query intra dist:", results_df["query_intra_mean"].mean())
    print("Mean query inter dist:", results_df["query_inter_mean"].mean())
    print("Mean query dist ratio:", results_df["query_dist_ratio"].mean())

    out_path = os.path.join(SAVE_DIR, "balanced_baseline_cnn_feature_fusion_loso_results.csv")
    results_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return results_df

# ============================================================
# RUN
# ============================================================

# cnn_only_results = run_loso_cnn_only()
fusion_results   = run_loso_cnn_feature_fusion()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEVICE: cuda
Original X: (2371, 1000, 8)
Tasks: ['Arithmetic' 'Stroop']
Labels: ['highlevel' 'lowlevel' 'midlevel' 'natural']
Subjects: [ 1  2  3  5  6  7  8  9 10 11 12 13 14 15]

After Arithmetic non-natural + drop subject 7:
X: (877, 1000, 8)
Subjects: [ 1  2  3  5  6  8  9 10 11 12 13 14 15]
Label counts: {'lowlevel': 240, 'midlevel': 261, 'highlevel': 376}

BALANCING EACH SUBJECT TO ITS MIN CLASS COUNT
Subject 1: raw counts={0: 21, 1: 22, 2: 24} -> keeping 21/class
Subject 2: raw counts={0: 18, 1: 19, 2: 34} -> keeping 18/class
Subject 3: raw counts={0: 13, 1: 19, 2: 22} -> keeping 13/class
Subject 5: raw counts={0: 13, 1: 20, 2: 20} -> keeping 13/class
Subject 6: raw counts={0: 16, 1: 18, 2: 25} -> keeping 16/class
Subject 8: raw counts={0: 35, 1: 34, 2: 40} -> keeping 34/class
Subject 9: raw counts={0: 17, 1: 17, 2: 55} -> keeping 17/class
Subject 10: 

,subject,lowlevel,midlevel,highlevel,total
0,1,21,21,21,63
1,2,18,18,18,54
2,3,13,13,13,39
3,5,13,13,13,39
4,6,16,16,16,48
5,8,34,34,34,102
6,9,17,17,17,51
7,10,14,14,14,42
8,11,21,21,21,63
9,12,16,16,16,48



Loading cached balanced handcrafted features...
Feature matrix: (702, 57)

RUNNING BALANCED BASELINE 2: CNN + HANDCRAFTED FEATURE FUSION

####################################################################################################
FOLD 1/13 | HELD OUT SUBJECT = 1
####################################################################################################
Train windows: 639
Held-out windows: 63
Train label counts: {'lowlevel': 213, 'midlevel': 213, 'highlevel': 213}
Held label counts : {'lowlevel': 21, 'midlevel': 21, 'highlevel': 21}
  Few-shot split:
    class 0 (lowlevel): total=21, support=5, query=16, support%=23.8%
    class 1 (midlevel): total=21, support=5, query=16, support%=23.8%
    class 2 (highlevel): total=21, support=5, query=16, support%=23.8%
Majority baseline query acc=0.3333 | balanced_acc=0.3333
[[16  0  0]
 [16  0  0]
 [16  0  0]]
Epoch 01/15 | loss=1.0972 | train_acc=0.3443 | train_bacc=0.3474 | zero_shot=0.3333 | zs_bacc=0.3333 | fewshot=0.4792 | 

,subject,split,intra_mean,inter_mean,ratio,intra_low,intra_mid,intra_high,inter_low_mid,inter_low_high,inter_mid_high
0,1,train,0.714498,0.962743,1.347440,0.765530,0.554023,0.823942,0.769796,1.130083,0.988350
1,1,query,0.743943,0.751977,1.010799,0.870695,0.669270,0.691863,0.790995,0.791004,0.673931
2,1,support,0.713532,0.732587,1.026705,1.006345,0.530192,0.604060,0.824335,0.830769,0.542657
3,2,train,1.017602,1.176446,1.156096,0.989239,1.014974,1.048593,1.240158,1.197939,1.091241
4,2,query,1.017621,0.995929,0.978683,0.972257,1.005762,1.074846,0.957083,1.000654,1.030050
5,2,support,0.832023,0.856212,1.029073,1.074070,0.886495,0.535506,0.891234,0.861101,0.816302
6,3,train,0.796763,0.967564,1.214369,1.005590,0.762056,0.622642,0.978995,1.089148,0.834549
7,3,query,0.383330,0.513689,1.340069,0.499548,0.195137,0.455306,0.530218,0.458162,0.552687
8,3,support,0.320087,0.474847,1.483492,0.516907,0.166704,0.276652,0.582386,0.394281,0.447874
9,5,train,0.809966,1.071924,1.323418,0.922401,0.667424,0.840072,1.143305,1.168475,0.903991


Saved: /content/drive/MyDrive/eeg_qc_outputs/CNN_fusion_distance_summary.csv

BALANCED CNN + FEATURE FUSION LOSO SUMMARY


,subject,best_zero_shot,best_zero_shot_bacc,best_fewshot,best_fewshot_bacc,majority_acc,majority_bacc,n_train,n_support,n_query,support_percent_total,query_intra_mean,query_inter_mean,query_dist_ratio
0,1,0.375000,0.375000,0.520833,0.520833,0.333333,0.333333,639,15,48,23.809524,0.743943,0.751977,1.010799
1,2,0.461538,0.461538,0.461538,0.461538,0.333333,0.333333,648,15,39,27.777778,1.017621,0.995929,0.978683
2,3,0.541667,0.541667,0.791667,0.791667,0.333333,0.333333,663,15,24,38.461538,0.383330,0.513689,1.340069
3,5,0.625000,0.625000,0.541667,0.541667,0.333333,0.333333,663,15,24,38.461538,0.853886,0.923249,1.081231
4,6,0.515152,0.515152,0.363636,0.363636,0.333333,0.333333,654,15,33,31.250000,0.796960,0.796654,0.999615
5,8,0.367816,0.367816,0.448276,0.448276,0.333333,0.333333,600,15,87,14.705882,0.958355,0.956503,0.998067
6,9,0.444444,0.444444,0.638889,0.638889,0.333333,0.333333,651,15,36,29.411765,0.613580,0.623918,1.016850
7,10,0.481481,0.481481,0.481481,0.481481,0.333333,0.333333,660,15,27,35.714286,0.441737,0.426819,0.966229
8,11,0.458333,0.458333,0.500000,0.500000,0.333333,0.333333,639,15,48,23.809524,0.804079,0.821406,1.021549
9,12,0.515152,0.515152,0.515152,0.515152,0.333333,0.333333,654,15,33,31.250000,0.147954,0.155475,1.050834


Mean ZS:               0.46868790900621143
Mean ZS bacc:          0.46868790900621143
Mean FS:               0.527234486949341
Mean FS bacc:          0.527234486949341
Mean majority:         0.3333333333333333
Mean support %:        29.497509582053517
Mean query intra dist: 0.6705692605330393
Mean query inter dist: 0.6915661585636629
Mean query dist ratio: 1.0423885049256527
Saved: /content/drive/MyDrive/eeg_qc_outputs/balanced_baseline_cnn_feature_fusion_loso_results.csv
